# WAQ Comprehensive Analysis — NetCDF-based

Reads pre-converted `.map.nc` files (built by `WAQ_map_to_netcdf` notebook)
instead of binary `.map` files. The conversion has been validated, so this
notebook can focus on analysis. `.his` files are still used for Secchi
(SecchiDept at CAB_averaged) and the surface-layer breathing depth (CAB at
the active surface).

**Corrections (12 Oct 2026)** — see the cells marked `CHANGED`:
1. Radiation files for FPV_03/FPV_04 point to the rebuilt ×0.65 files the runs used; `.his` paths are given
   per scenario (03/04 sit in their run folders).
2. FPV_04 difference panel is now drawn in the Section 10 depth grid.
3. TotN / TotP use one definition everywhere, without sediment (`…S1`) substances or AAP:
   TotN = NH4 + NO3 + PON1 + algal N, TotP = PO4 + POP1 + algal P (algal N/P from the BLOOM `.spe`
   NCRALG/PCRALG of each type).
4. Chl a = Σ biomass × CHLACALG (`.spe`) × 1000 — unchanged ratios, now checked against `.his` `Chlfa` (Section 6.0).
5. BLOOM `Limit*` outputs are read as on/off flags (fraction of time each limit binds), not as 0–1 factors.
6. The "depth-averaged" series are labelled **layer mean (K=7–15)**: the mean over all wet cells of K=7–15,
   not volume-weighted.
7. The analysis window includes the whole of 31 August; the `.his` Secchi figures all use `CAB_averaged`.
8. Spatial maps of quantities that are not in the `.map` (Chl a, total algae, TotN, TotP) — Section 11.
9. New Section 12: **shading-only test** — baseline hydrodynamics with the FPV_02 light and wind (run 08).
10. Removed leftover/diagnostic cells that belonged to other notebooks.

**Inputs:**
- 4 `.map.nc` files (one per scenario, alongside the `.map` files)
- 4 `.his` files (for Secchi + surface breathing)
- 4 radiation `.bin` files (define the FPV_02 perimeter)
- run 08 (shading-only test): one more `.map.nc` and `.his` (Section 12)

**Outputs saved to** `RESULTS_DIR`:
- All scenario figures (PNG)
- Perimeter time-series CSVs (daily + monthly)

**Convention reminders:**
- Layer K=0 is the top of the model (mostly dry); K=7..15 are the active
  reservoir body. The depth axis runs from 0 m (top of K=7) to ~8.8 m.
- FPV_02 perimeter is the (M, N) columns covered by the FPV_02 array,
  projected from the surface radiation flag down through all K layers.


## Section 1 — Imports & user configuration

In [ ]:
import os
import struct
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import Normalize, SymLogNorm
from matplotlib.patches import Patch

# ── Paths ────────────────────────────────────────────────────────────────────
BIN_DIR  = r'//luna/fst/LEC/Users/midauarg/delft_modelling/wq_0.1min_secondfolder'
MAP_DIR  = os.path.join(BIN_DIR, 'map_files_scenarios_remmaped')
HIS_DIR  = os.path.join(BIN_DIR, 'his_scenario_files_remmaped')

RESULTS_DIR = os.path.join(BIN_DIR, 'results_scenarios_remmaped_all')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Pre-converted NetCDFs (one per scenario)
NC_FILES = {
    'Baseline': os.path.join(MAP_DIR, 'Torun_WAQ_Apr2023.nc'),
    'FPV_02':   os.path.join(MAP_DIR, 'Torun_WAQ_Apr2023_scen02.nc'),
    'FPV_03':   os.path.join(MAP_DIR, 'Torun_WAQ_Apr2023_scen03.nc'),
    'FPV_04':   os.path.join(MAP_DIR, 'Torun_WAQ_Apr2023_scen04.nc'),
}
# CHANGED: full paths (03/04 .his are in their run folders)
HIS_FILES = {
    'Baseline': os.path.join(HIS_DIR, 'Torun_WAQ_Apr2023.his'),
    'FPV_02':   os.path.join(HIS_DIR, 'Torun_WAQ_Apr2023_scen02.his'),
    'FPV_03':   os.path.join(BIN_DIR, '03scen_fixed', 'Torun_WAQ_Apr2023_scen03.his'),
    'FPV_04':   os.path.join(BIN_DIR, '04scen_fixed', 'Torun_WAQ_Apr2023_scen04.his'),
}

# Shading-only test (run 08): baseline hydrodynamics + FPV_02 light and wind. Section 12 only.
SHADE = 'ShadeOnly_02'
SHADE_LABEL = 'Shading only (FPV_02 light & wind, baseline hydrodynamics)'
SHADE_NC  = os.path.join(MAP_DIR, 'Torun_WAQ_Apr2023_scen08.nc')
SHADE_HIS = os.path.join(BIN_DIR, '08_baselinewithSRWSfrom02', 'Torun_WAQ_Apr2023_scen08.his')
RAD_BIN = {
    'noFPV': 'waq-sw_radiation_flux-noFPV_remapped_upper.bin',
    '02':    'waq-daily-sw_radiation_flux-02towaq_fpv41_nogaps_remapped_upper.bin',
    # CHANGED: the rebuilt x0.65 files the 03/04 runs used
    '03':    'waq-daily-sw_radiation_flux-03towaq_fpv41_10x10gaps_remapped_upper_F065_rebuilt.bin',
    '04':    'waq-daily-sw_radiation_flux-04towaq_fpv41_donutgap_remapped_upper_F065_rebuilt.bin',
}

# ── Constants ────────────────────────────────────────────────────────────────
N_SEG          = 10999     # active WAQ segments (radiation .bin layout)
T0             = pd.Timestamp('2022-09-09')
ANALYSIS_START = pd.Timestamp('2023-04-01')
ANALYSIS_END   = pd.Timestamp('2023-08-31')
ANALYSIS_END_INCL = ANALYSIS_END + pd.Timedelta(hours=23, minutes=59, seconds=59)   # CHANGED: whole last day
RAD_NT         = 554

K_TOP    = 7
K_BOT    = 15
K_ACTIVE = K_BOT - K_TOP + 1

# Z-model thicknesses from .mdf Thick. Same as in the NetCDF coord, but kept
# here for plotting axes.
THICK_PCT_FLOW = [12.5, 6.25, 6.25, 6.25, 6.25, 6.25, 6.25, 6.25,
                  3.125, 3.125, 3.125, 3.125, 3.125, 3.125, 12.5, 12.5]
Z_TOTAL = 14.79
THICKNESS_M = {waq_k: THICK_PCT_FLOW[15 - waq_k] / 100.0 * Z_TOTAL
               for waq_k in range(16)}

# ── User-editable configuration ──────────────────────────────────────────────
SUBSTANCES = ['OXY', 'NH4', 'NO3', 'PO4', 'Si',
              'POC1', 'PON1', 'POP1',
              'FDIATOMS_E', 'GREENS_E', 'MICROCYS_E', 'FFLAGELA']

SNAPSHOT_DATES = ['2023-05-15', '2023-06-15', '2023-07-15', '2023-08-14']

# Chl a : C ratio per type (CHLACALG in the BLOOM .spe, gChl/gC). Chl a (µg/L) = Σ biomass × CHLA_C × 1000
CHLA_C = {
    'FDIATOMS_E': 0.025, 'FDIATOMS_P': 0.025,
    'FFLAGELA':   0.029,
    'GREENS_E':   0.025, 'GREENS_N':   0.025, 'GREENS_P':   0.025,
    'MICROCYS_E': 0.025, 'MICROCYS_N': 0.017, 'MICROCYS_P': 0.017,
}
ALGAE = list(CHLA_C.keys())

# CHANGED: N:C and P:C per type (NCRALG, PCRALG in the BLOOM .spe, gN/gC and gP/gC) for algal N and P
NCR_ALG = {'FDIATOMS_E': 0.175, 'FDIATOMS_P': 0.155, 'FFLAGELA': 0.275,
           'GREENS_E': 0.275, 'GREENS_N': 0.175, 'GREENS_P': 0.200,
           'MICROCYS_E': 0.225, 'MICROCYS_N': 0.113, 'MICROCYS_P': 0.175}
PCR_ALG = {'FDIATOMS_E': 0.0180, 'FDIATOMS_P': 0.0113, 'FFLAGELA': 0.0180,
           'GREENS_E': 0.0238, 'GREENS_N': 0.0150, 'GREENS_P': 0.0125,
           'MICROCYS_E': 0.0300, 'MICROCYS_N': 0.0275, 'MICROCYS_P': 0.0225}

# CHANGED: label for the K=7..15 cell mean (not volume-weighted)
LAYERMEAN_LABEL = 'Layer mean (K=7–15)'

SPECIES_GROUPS = {
    'Diatoms':       ['FDIATOMS_E', 'FDIATOMS_P'],
    'Flagellates':   ['FFLAGELA'],
    'Greens':        ['GREENS_E', 'GREENS_N', 'GREENS_P'],
    'Cyanobacteria': ['MICROCYS_E', 'MICROCYS_N', 'MICROCYS_P'],
}
GROUP_COLORS = {'Diatoms':'#ff7f0e', 'Flagellates':'#1f77b4',
                'Greens':'#2ca02c', 'Cyanobacteria':'#9467bd'}

SCEN_COLORS = {'Baseline': 'k', 'FPV_02': '#1f77b4',
               'FPV_03': '#ff7f0e', 'FPV_04': '#2ca02c',
               'ShadeOnly_02': '#7f7f7f'}
SCEN_ORDER  = ['Baseline', 'FPV_02', 'FPV_03', 'FPV_04']

OXY_MAX_PHYS = 20.0

SEASONS = {
    'Spring (Apr-May)': ('2023-04-01', '2023-05-31'),
    'Summer (Jun-Aug)': ('2023-06-01', '2023-08-31'),
    'Apr-Aug':          ('2023-04-01', '2023-08-31'),
}

# Plot save settings
SAVE_FIGS = True
FIG_DPI   = 150
FIG_FMT   = 'png'

def save_fig(fig, filename):
    if not SAVE_FIGS:
        return None
    path = os.path.join(RESULTS_DIR, f'{filename}.{FIG_FMT}')
    fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight')
    return path

print('Configuration loaded.')
print(f'  Analysis window:  {ANALYSIS_START.date()} → {ANALYSIS_END.date()}')
print(f'  Active layers:    K={K_TOP}..{K_BOT}')
print(f'  Substances:       {", ".join(SUBSTANCES)}')
print(f'  Results folder:   {RESULTS_DIR}')

# ── Substance display labels (full name + units) ────────────────────────────
# Used by every plotting function for ylabels, titles, and colorbars.
SUBSTANCE_LABELS = {
    'OXY':        'Dissolved Oxygen (g/m³)',
    'NH4':        'Ammonium NH4 (gN/m³)',
    'NO3':        'Nitrate NO3 (gN/m³)',
    'PO4':        'Phosphate PO4 (gP/m³)',
    'Si':         'Silicate Si (gSi/m³)',
    'POC1':       'Particulate Organic\nCarbon POC1 (gC/m³)',
    'PON1':       'Particulate Organic\nNitrogen PON1 (gN/m³)',
    'POP1':       'Particulate Organic\nPhosphorus POP1 (gP/m³)',
    'AAP':        'Adsorbed Apatite\nPhosphorus AAP (gP/m³)',
    'Opal':       'Biogenic Silicate\nOpal (gSi/m³)',
    'TotN':       'Total Nitrogen TotN (gN/m³)',
    'TotP':       'Total Phosphorus TotP (gP/m³)',
    'TotAlgae':   'Total Algae biomass (gC/m³)',
    'AlgN':       'Algal nitrogen (gN/m³)',
    'AlgP':       'Algal phosphorus (gP/m³)',
    'Chlfa':      'Chlorophyll a (µg/L)',
    'FDIATOMS_E': 'Diatoms — E ecotype (gC/m³)',
    'FDIATOMS_N': 'Diatoms — N ecotype (gC/m³)',
    'FDIATOMS_P': 'Diatoms — P ecotype (gC/m³)',
    'FFLAGELA':   'Flagellates (gC/m³)',
    'GREENS_E':   'Greens — E ecotype (gC/m³)',
    'GREENS_N':   'Greens — N ecotype (gC/m³)',
    'GREENS_P':   'Greens — P ecotype (gC/m³)',
    'MICROCYS_E': 'Microcystis — E ecotype (gC/m³)',
    'MICROCYS_N': 'Microcystis — N ecotype (gC/m³)',
    'MICROCYS_P': 'Microcystis — P ecotype (gC/m³)',
    'SecchiDept': 'Secchi depth (m)',
    'RadAve':     'Average radiation (W/m²)',
    'Temp':       'Temperature (°C)',
    'VWind':      'Wind speed (m/s)',
    'ExtVl':      'Extinction coefficient (1/m)',
}

def label_for(sub):
    """Return display label for a substance — fallback to the raw name."""
    return SUBSTANCE_LABELS.get(sub, sub)


# ── Panel-letter annotation helper ──────────────────────────────────────────
# Standardised (a), (b), (c)... letters placed inside the top-left corner of
# each axes, on a small white background patch for legibility against any
# colormap. Use INSTEAD of axes titles in multi-panel figures.
def add_panel_letter(ax, letter, x=0.02, y=0.97, fontsize=10):
    """Annotate an axes with a panel letter like '(a)' in the top-left.

    Args:
        ax: matplotlib axes.
        letter: text to display (e.g. '(a)', 'a)' or 'A').
        x, y: position in axes fraction (0-1).
        fontsize: text size.
    """
    ax.text(x, y, letter, transform=ax.transAxes,
            fontsize=fontsize,
            ha='left', va='top',
            bbox=dict(facecolor='white', edgecolor='none',
                      pad=2.0, alpha=0.8))

## Section 2 — Open NetCDFs

Each scenario's data is opened lazily by xarray. The trim-to-window operation
is also lazy — only the timesteps in `ANALYSIS_START..ANALYSIS_END` are
loaded when used.

In [ ]:
import time

def open_scenario(path):
    """Three-stage open with timing — surface the actual bottleneck."""
    t0 = time.time()
    ds = xr.open_dataset(path)
    t_open = time.time() - t0

    t0 = time.time()
    ds = ds.sel(time=slice(ANALYSIS_START, ANALYSIS_END_INCL))   # CHANGED: whole last day
    t_slice = time.time() - t0

    t0 = time.time()
    # ds = ds.resample(time='D').mean()
    # Force the lazy resample to actually execute
    _ = ds[list(ds.data_vars)[0]].values
    t_resample = time.time() - t0

    return ds, (t_open, t_slice, t_resample)


print('Opening scenario NetCDFs (with timing breakdown)...\n')
runs = {}
for scen, path in NC_FILES.items():
    print(f'  {scen}...', flush=True)
    t_total = time.time()
    runs[scen], (t_open, t_slice, t_resample) = open_scenario(path)
    elapsed = time.time() - t_total
    print(f'    open_dataset:  {t_open:6.1f}s', flush=True)
    print(f'    sel(time):     {t_slice:6.1f}s', flush=True)
    print(f'    resample(D):   {t_resample:6.1f}s', flush=True)
    print(f'    TOTAL:         {elapsed:6.1f}s', flush=True)

In [ ]:
print('Scenarios loaded:', list(runs.keys()))
for scen, ds in runs.items():
    print(f'  {scen}: time {ds.time.values[0]} → {ds.time.values[-1]}, '
          f'{len(ds.time)} timesteps, {len(ds.data_vars)} variables')

## Section 3 — Read `.his` for Secchi + surface breathing

`.his` is still needed for substances not in the `.map` (and therefore not in
the NetCDF): `SecchiDept`, `Depth`, etc. Layout is `(n_loc, n_sub)` per
record.

In [ ]:
def read_his_file(path, t0=T0):
    """Read DELWAQ .his with (n_loc, n_sub) per-record layout."""
    with open(path, 'rb') as f:
        titles = [f.read(40).decode('latin-1', errors='replace').strip()
                  for _ in range(4)]
        n_sub, n_loc = struct.unpack('<ii', f.read(8))
        substances = [f.read(20).decode('latin-1', errors='replace').strip()
                      for _ in range(n_sub)]
        locations = []
        for _ in range(n_loc):
            _ = struct.unpack('<i', f.read(4))[0]
            name = f.read(20).decode('latin-1', errors='replace').strip()
            locations.append(name)
        header_size = f.tell()
        record_size = 4 + n_sub * n_loc * 4
        fsize = os.path.getsize(path)
        n_times = (fsize - header_size) // record_size

        data = np.empty((n_times, n_loc, n_sub), dtype=np.float32)
        timestamps = []
        f.seek(header_size)
        for i in range(n_times):
            ts = struct.unpack('<i', f.read(4))[0]
            data[i] = np.frombuffer(f.read(n_sub * n_loc * 4),
                                    dtype='<f4').reshape(n_loc, n_sub)
            timestamps.append(t0 + pd.Timedelta(seconds=ts))

    return {'substances': substances,
            'idx':        {s: i for i, s in enumerate(substances)},
            'locations':  locations,
            'loc_idx':    {l: i for i, l in enumerate(locations)},
            'data':       data,
            'timestamps': pd.DatetimeIndex(timestamps)}

print('Loading .his files...')
his_runs = {}
for name, fname in HIS_FILES.items():
    his_runs[name] = read_his_file(fname)        # CHANGED: HIS_FILES holds full paths

# Trim to window AND resample to daily means (consistent with the NetCDF
# reads in Section 2). After this, every .his data access yields daily values.
for name in list(his_runs.keys()):
    r = his_runs[name]
    # 1) Window trim
    mask = (r['timestamps'] >= ANALYSIS_START) & (r['timestamps'] <= ANALYSIS_END_INCL)
    idx = np.where(mask)[0]
    data_win = r['data'][idx]
    ts_win   = r['timestamps'][idx]

    # 2) Daily resample. Build a DataFrame per location-substance pair via a
    #    flattened (time, loc*sub) array, resample, reshape back.
    n_t, n_loc, n_sub = data_win.shape
    flat = data_win.reshape(n_t, n_loc * n_sub)
    df_flat = pd.DataFrame(flat, index=ts_win)
    df_daily = df_flat.resample('D').mean()
    data_daily = df_daily.values.reshape(len(df_daily), n_loc, n_sub).astype(np.float32)
    ts_daily = pd.DatetimeIndex(df_daily.index)

    his_runs[name] = {**r,
                       'data':       data_daily,
                       'timestamps': ts_daily}
    print(f'  {name}: {len(ts_daily)} daily steps in window '
          f'(was {len(idx)} native-resolution steps)')

# CAB monitor lookups
locs = his_runs['Baseline']['locations']
def cab_loc_for_k(locations, k_python):
    target = f'CAB ({k_python + 1})'   # .his uses 1-based; WAQ K is 0-based
    for li, name in enumerate(locations):
        if name == target:
            return li
    return None

CAB_K = {k: cab_loc_for_k(locs, k) for k in range(K_TOP, K_BOT + 1)}
CAB_K = {k: v for k, v in CAB_K.items() if v is not None}
CAB_AVERAGED_IDX = locs.index('CAB_averaged') if 'CAB_averaged' in locs else None
print(f'\nCAB indices per K (from .his): {CAB_K}')
print(f'CAB_averaged index:               {CAB_AVERAGED_IDX}')

In [ ]:
# Step 3.5: Load radiation .bin files (needed for .bin-based perimeter detection)
def load_rad_bin(path, n_seg=N_SEG, n_t=RAD_NT):
    out = np.empty((n_t, n_seg), np.float32)
    with open(path, 'rb') as f:
        for i in range(n_t):
            tb = f.read(4)
            if len(tb) < 4:
                out = out[:i]; break
            out[i] = np.frombuffer(f.read(n_seg * 4), dtype='<f4')
    return out

print(f'\nLoading radiation .bin files...')
rad = {k: load_rad_bin(os.path.join(BIN_DIR, v)) for k, v in RAD_BIN.items()}
print(f'  rad keys: {list(rad.keys())}')
print(f'  rad shapes: {[(k, v.shape) for k, v in rad.items()]}')

## Section 4 — Layer mid-depths + surface breathing

Fixed mid-depths from `.mdf` Thick (deep layers don't breathe in Z-model).
Surface mid-depth from `.his` CAB(8) Depth (time-varying).

In [ ]:
LAYER_MID_DEPTHS = {}
cum = 0.0
for k in range(K_TOP, K_BOT + 1):
    LAYER_MID_DEPTHS[k] = cum + THICKNESS_M[k] / 2.0
    cum += THICKNESS_M[k]

# Surface (K=7) thickness time series from .his CAB(8) Depth
THICKNESS_MAX_PHYS = 15.0
depth_si = his_runs['Baseline']['idx']['Depth']
surface_thickness_t = his_runs['Baseline']['data'][:, CAB_K[K_TOP], depth_si].astype(float)
surface_thickness_t[surface_thickness_t <= 0] = np.nan
surface_thickness_t[surface_thickness_t > THICKNESS_MAX_PHYS] = np.nan
LAYER_MID_DEPTHS[K_TOP] = 0.5 * np.nanmedian(surface_thickness_t)

surface_mid_depth_series = pd.Series(0.5 * surface_thickness_t,
                                     index=his_runs['Baseline']['timestamps'])

print('Mid-depths for Hovmöller y-axis:')
for k in range(K_TOP, K_BOT + 1):
    print(f'  K={k:>2}  mid-depth = {LAYER_MID_DEPTHS[k]:>6.3f} m  '
          f'(thickness {THICKNESS_M[k]:.3f} m)')
total = sum(THICKNESS_M[k] for k in range(K_TOP, K_BOT + 1))
print(f'\nTotal column thickness K={K_TOP}..{K_BOT}: {total:.2f} m')
print(f'Surface (K={K_TOP}) breathing range: '
      f'{surface_mid_depth_series.min():.3f} → '
      f'{surface_mid_depth_series.max():.3f} m')

## Section 5 — Define FPV_02 perimeter as (M, N) mask

The radiation binaries flag covered segments at the model's *surface* layer.
We translate this into a 2D `(M, N)` perimeter mask that the NetCDF can use
to select columns under the FPV_02 array.

The FPV_02 radiation file changes 622 surface segments, which map one-to-one onto 622 (M, N) columns.
The same FPV_02 footprint is used as the averaging area for **every** scenario, so all scenarios are averaged
over the same water. FPV_03 (684 columns) and FPV_04 (651) have different footprints: their averages include
their gaps and miss parts of their own arrays.

In [ ]:
# ── Load segnum and build new_to_old from .vol file ─────────────────────
# segnum: (M, N, K) → FLOW segment ID (1-based), from Tecplot .mat export.
# new_to_old: compact_idx → FLOW segment ID (0-based), built from .vol.
# Then rebuild perim_da using .bin-based detection (true FPV footprint).

# import os
# import numpy as np
# from scipy.io import loadmat

# N_SEG_IN  = 27904     # FLOW segments before cut
# N_SEG_BIN = 10999     # compact ever-active segments

# # Step 1: load segnum from the Tecplot .mat export
# SEGNUM_PATH = '//luna/fst/LEC/Users/midauarg/delft_modelling/wq_0.1min_secondfolder/segment number.mat'
# mat = loadmat(SEGNUM_PATH)
# segnum = mat['data'][0, 0]['Val']
# print(f'segnum loaded: shape {segnum.shape}, '
#       f'NaN cells {np.sum(np.isnan(segnum))}, '
#       f'max FLOW ID {int(np.nanmax(segnum))}')


# # Step 2: build ever-active mask from baseline original .vol
# SCEN_VOL_ORIG = '//luna/fst/LEC/Users/midauarg/delft_modelling/wq_0.1min_secondfolder/com_files/com-WQexport_30min_fullperiod.vol'
# rec_size_s = (N_SEG_IN + 1) * 4
# fsize      = os.path.getsize(SCEN_VOL_ORIG)
# n_times    = fsize // rec_size_s

# print(f'\nReading baseline .vol ({fsize/1e6:.0f} MB, {n_times} timesteps)...')
# ever_active = np.zeros(N_SEG_IN, dtype=bool)
# with open(SCEN_VOL_ORIG, 'rb') as f:
#     for i in range(n_times):
#         f.read(4)
#         raw = np.frombuffer(f.read(N_SEG_IN * 4), dtype='<f4').copy()
#         ever_active |= (raw > 0)
#         if (i + 1) % 2000 == 0:
#             print(f'  {i+1:,}/{n_times:,}...', flush=True)

# all_active_idx = np.where(ever_active)[0]
# new_to_old = all_active_idx[:N_SEG_BIN]
# print(f'  Total ever-active: {len(all_active_idx)}')
# print(f'  new_to_old: length {len(new_to_old)}')


# # Step 3: sanity-check against CAB (M=24, N=37, K=7) → compact index 1153
# target_flow_seg = int(segnum[24, 37, 7]) - 1   # 0-based
# matching = np.where(new_to_old == target_flow_seg)[0]
# print(f'\nCAB cross-check: FLOW seg {target_flow_seg+1} → '
#       f'compact index {matching} (expected: 1153)')


# # Step 4: build perim_da using .bin-based detection (TRUE FPV footprint)
# print(f'\nBuilding perim_da using .bin radiation differences...')
# diff_02 = rad['02'] - rad['noFPV']
# covered_surf_segs = np.where((diff_02 < -1e-6).any(axis=0))[0]
# print(f'  FPV_02 covers {len(covered_surf_segs)} compact segments')

# fpv_mask_mn = np.zeros((segnum.shape[0], segnum.shape[1]), dtype=bool)
# for compact_idx in covered_surf_segs:
#     flow_seg = new_to_old[compact_idx] + 1
#     found = False
#     for m_ in range(segnum.shape[0]):
#         for n_ in range(segnum.shape[1]):
#             v = segnum[m_, n_, K_TOP]
#             if not np.isnan(v) and int(v) == flow_seg:
#                 fpv_mask_mn[m_, n_] = True
#                 found = True
#                 break
#         if found:
#             break

# perim_da = xr.DataArray(fpv_mask_mn,
#                          coords=[runs['Baseline'].M, runs['Baseline'].N],
#                          dims=['M', 'N'])
# n_cells = int(perim_da.sum())
# print(f'\nNew perim_da: {n_cells} cells')
# if n_cells == len(covered_surf_segs):
#     print(f'  ✓ Matches compact-segment count')
# else:
#     print(f'  ⚠ Mismatch — expected {len(covered_surf_segs)}')


# # Step 5: save lookup arrays for future kernel sessions
# LOOKUP_PATH = '//luna/fst/LEC/Users/midauarg/delft_modelling/wq_0.1min_secondfolder/segment_lookup.npz'
# np.savez(LOOKUP_PATH, segnum=segnum, new_to_old=new_to_old,
#          fpv_mask_mn=fpv_mask_mn)
# print(f'\nSaved segnum, new_to_old, and fpv_mask_mn to:\n  {LOOKUP_PATH}')
# print(f'On future runs, you can load these directly with np.load() and skip '
#       f'the .vol parsing.')

In [ ]:
lookup = np.load('//luna/fst/LEC/Users/midauarg/delft_modelling/wq_0.1min_secondfolder/segment_lookup.npz')
segnum = lookup['segnum']
new_to_old = lookup['new_to_old']

In [ ]:
def load_rad_bin(path, n_seg=N_SEG, n_t=RAD_NT):
    out = np.empty((n_t, n_seg), np.float32)
    with open(path, 'rb') as f:
        for i in range(n_t):
            tb = f.read(4)
            if len(tb) < 4:
                out = out[:i]; break
            out[i] = np.frombuffer(f.read(n_seg * 4), dtype='<f4')
    return out

rad = {k: load_rad_bin(os.path.join(BIN_DIR, v)) for k, v in RAD_BIN.items()}
base_rad = rad['noFPV']

# Surface segments where FPV_02 attenuates radiation
diff_02 = rad['02'] - base_rad
covered_surf_segs = np.where((diff_02 < -1e-6).any(axis=0))[0]
print(f'FPV_02 covers {len(covered_surf_segs)} compact segments')

# ── Build perim_da from segnum + new_to_old (true FPV footprint) ────────
# Requires segnum and new_to_old to be in scope (from the lookup cell).
print('Building perim_da from compact segments → (M, N) at K=K_TOP...')

fpv_mask_mn = np.zeros((segnum.shape[0], segnum.shape[1]), dtype=bool)
for compact_idx in covered_surf_segs:
    flow_seg = new_to_old[compact_idx] + 1   # 1-based
    found = False
    for m_ in range(segnum.shape[0]):
        for n_ in range(segnum.shape[1]):
            v = segnum[m_, n_, K_TOP]
            if not np.isnan(v) and int(v) == flow_seg:
                fpv_mask_mn[m_, n_] = True
                found = True
                break
        if found:
            break

perim_da = xr.DataArray(fpv_mask_mn,
                         coords=[runs['Baseline'].M, runs['Baseline'].N],
                         dims=['M', 'N'])
print(f'\nperim_da: {int(perim_da.sum())} (M, N) cells '
      f'at K=K_TOP={K_TOP} surface')
print(f'  (compact-segment count: {len(covered_surf_segs)})')

In [ ]:
# Validate perimeter by counting cells per layer in the FPV_02 footprint.
# In the NetCDF, K=7..15 each have a different non-NaN-cell count.
# Mask cells × non-NaN per layer should match.
print('Perimeter cell counts per active layer (FPV_02):')
for k in range(K_TOP, K_BOT + 1):
    valid = runs['FPV_02']['OXY'].sel(K=k).isel(time=0).notnull()
    n_in_perim = int((valid & perim_da).sum())
    n_total    = int(valid.sum())
    print(f'  K={k:2d}: {n_in_perim:4d} cells in perimeter / {n_total:5d} total')

## Section 6 — Perimeter-averaged time series (daily + monthly)

For each scenario: compute the FPV_02-perimeter mean of each substance per
timestep over active layers (K=7..15), add derived fields (TotAlgae, Chlfa,
TotN, TotP), aggregate to monthly means, and save daily + monthly CSVs.

With xarray, the "perimeter mean over active layers" is a one-liner: ds[sub].where(perim_da).sel(K=slice(K_TOP, K_BOT)).mean(['M', 'N', 'K'])

### 6.0 — Check: Chl a from biomass × CHLACALG against BLOOM's own `Chlfa` (CHANGED, new)
`Chlfa` is not in the `.map`, so every Chl a value away from the `.his` stations is computed as
Σ biomass × CHLACALG × 1000 with the `.spe` ratios in `CHLA_C`. The `.his` holds both the algal biomass and
BLOOM's `Chlfa` at CAB and BIS, so the formula can be checked there for every run. A relative error near 0 %
means the maps and profiles show the same Chl a as BLOOM.

In [ ]:
# Compare Σ biomass × CHLACALG × 1000 with .his Chlfa at all CAB/BIS layer monitors
def chla_check(his_run):
    locs = [i for i, l in enumerate(his_run['locations']) if 'averaged' not in l.lower()]
    alg = [s for s in ALGAE if s in his_run['idx']]
    if 'Chlfa' not in his_run['idx'] or not alg:
        return None
    X = np.stack([his_run['data'][:, locs, his_run['idx'][s]] for s in alg], axis=-1).astype(float)
    calc = (X * np.array([CHLA_C[s] for s in alg])).sum(axis=-1) * 1000.0
    his = his_run['data'][:, locs, his_run['idx']['Chlfa']].astype(float)
    ok = np.isfinite(calc) & np.isfinite(his) & (his > 1.0)
    rel = np.abs(calc[ok] - his[ok]) / his[ok] * 100
    return dict(n=int(ok.sum()), median=np.median(rel), p95=np.percentile(rel, 95), max=rel.max(),
                ratio=np.median(calc[ok] / his[ok]))

print('Chl a check (daily means, all CAB/BIS layer monitors, where Chlfa > 1 µg/L):')
print(f'  {"Run":<14} {"n":>6} {"median err %":>13} {"95th pct %":>11} {"max %":>8} {"calc/his":>9}')
for name, r in his_runs.items():
    c = chla_check(r)
    if c is None:
        print(f'  {name:<14} Chlfa or algae not in .his'); continue
    print(f'  {name:<14} {c["n"]:>6} {c["median"]:>13.2f} {c["p95"]:>11.2f} {c["max"]:>8.2f} {c["ratio"]:>9.4f}')

In [ ]:
def perimeter_mean_series(ds, perim_da, k_slice=slice(K_TOP, K_BOT)):
    masked = ds.where(perim_da).sel(K=k_slice)
    avg = masked.mean(['M', 'N', 'K'])
    avg = avg.resample(time='D').mean()    # daily resample after reduction
    df = avg.to_dataframe()
    return df

def add_derived(df):
    """CHANGED. Add TotAlgae, Chlfa, AlgN, AlgP, TotN, TotP (one definition for every layer and section).
    - Chlfa = sum(biomass x CHLACALG) x 1000                     (µg/L)
    - TotN  = NH4 + NO3 + PON1 + algal N  (algal N = sum(biomass x NCRALG))
    - TotP  = PO4 + POP1 + algal P        (algal P = sum(biomass x PCRALG))
    Sediment-layer substances (AAPS1, DetNS1, DetPS1 - mass per area) and AAP are not included."""
    df = df.copy()
    algae_present = [s for s in ALGAE if s in df.columns]
    if algae_present:
        df['TotAlgae'] = sum(df[s] for s in algae_present)
        df['Chlfa']    = sum(df[s] * CHLA_C[s] for s in algae_present) * 1000.0
        df['AlgN']     = sum(df[s] * NCR_ALG[s] for s in algae_present)
        df['AlgP']     = sum(df[s] * PCR_ALG[s] for s in algae_present)
    if all(c in df.columns for c in ['NH4', 'NO3', 'PON1']):
        df['TotN'] = df['NH4'] + df['NO3'] + df['PON1'] + df.get('AlgN', 0)
    if all(c in df.columns for c in ['PO4', 'POP1']):
        df['TotP'] = df['PO4'] + df['POP1'] + df.get('AlgP', 0)
    if 'OXY' in df.columns:
        df.loc[df['OXY'] > OXY_MAX_PHYS, 'OXY'] = np.nan
    return df

print('Computing daily perimeter means (xarray)...')
series_daily = {}
for scen, ds in runs.items():
    print(f'  {scen}...')
    series_daily[scen] = add_derived(perimeter_mean_series(ds, perim_da))

series_monthly = {scen: df.resample('MS').mean()
                  for scen, df in series_daily.items()}

# Save CSVs
for scen in SCEN_ORDER:
    series_daily[scen].to_csv(os.path.join(RESULTS_DIR, f'series_daily_{scen}.csv'))
    series_monthly[scen].to_csv(os.path.join(RESULTS_DIR, f'series_monthly_{scen}.csv'))
print(f'\nDaily + monthly CSVs saved to {RESULTS_DIR}')

# Monthly summary
KEY_SUBS = ['OXY', 'NH4', 'NO3', 'PO4', 'TotN', 'TotP', 'TotAlgae', 'Chlfa']
print('\nMonthly perimeter means:')
print('=' * 84)
print(f'{"Month":<10} {"Substance":<10} ' + ' '.join(f'{s:>10}' for s in SCEN_ORDER))
print('-' * 84)
for month in series_monthly['Baseline'].index:
    for sub in KEY_SUBS:
        if sub not in series_monthly['Baseline'].columns:
            continue
        vals = [series_monthly[scen].at[month, sub] for scen in SCEN_ORDER]
        print(f'{month.strftime("%Y-%m"):<10} {sub:<10} ' + ' '.join(f'{v:>10.4f}' for v in vals))
    print()

In [ ]:
# Monthly comparison plot — one figure per substance, FPV scenarios shown
# as bars with % delta vs Baseline (dashed reference line), one panel per
# month + Apr-Aug total. Matches the Image-2 style.

def label_for(sub):
    return SUBSTANCE_LABELS.get(sub, sub)


def plot_monthly_delta_panels(series_monthly, series_daily, subs,
                                save_name='monthly_delta_panels'):
    """One figure per substance. Panels = each month + a final Apr-Aug
    total panel. Per panel: Baseline as dashed horizontal reference, FPV
    scenarios as coloured bars labelled with their % change vs Baseline.
    """
    subs = [s for s in subs if s in series_monthly['Baseline'].columns]
    months = series_monthly['Baseline'].index
    fpv_scens = [s for s in SCEN_ORDER if s != 'Baseline']

    for sub in subs:
        # Period definitions: each month + Apr-Aug total
        periods = [(m.strftime('%b %Y'), m) for m in months]
        # Add Apr-Aug total as the final "panel"
        periods.append(('Apr-Aug 2023', None))   # None signals "use daily mean"

        n_panels = len(periods)
        fig, axes = plt.subplots(1, n_panels,
                                 figsize=(2.1 * n_panels, 3.5),
                                 sharey=True)
        if n_panels == 1:
            axes = [axes]

        for ax, (period_label, period_month) in zip(axes, periods):
            # Compute values for this period
            if period_month is None:
                # Apr-Aug total: mean over the whole daily series
                base_val = series_daily['Baseline'][sub].mean()
                fpv_vals = [series_daily[s][sub].mean() for s in fpv_scens]
            else:
                base_val = series_monthly['Baseline'][sub].loc[period_month]
                fpv_vals = [series_monthly[s][sub].loc[period_month]
                            for s in fpv_scens]

            # Bar plot for FPV scenarios
            x = np.arange(len(fpv_scens))
            colours = [SCEN_COLORS[s] for s in fpv_scens]
            bars = ax.bar(x, fpv_vals, width=0.7, color=colours,
                          edgecolor='black', lw=0.4)

            # Baseline as dashed reference line
            ax.axhline(base_val, color='k', lw=1.5, ls='--', label='Baseline')

            # % change labels on each bar (green if positive, red if negative)
            for j, (bar, val) in enumerate(zip(bars, fpv_vals)):
                if base_val == 0 or np.isnan(base_val) or np.isnan(val):
                    pct_str = 'n/a'
                    colour = 'grey'
                else:
                    pct = (val - base_val) / base_val * 100
                    pct_str = f'{pct:+.1f}%'
                    colour = '#2c7a2c' if pct >= 0 else '#c0392b'
                # Place label inside bar near top
                ax.annotate(pct_str,
                            xy=(bar.get_x() + bar.get_width() / 2, val),
                            xytext=(0, 4), textcoords='offset points',
                            ha='center', va='bottom', fontsize=9,
                            color=colour)

            ax.set_xticks(x)
            ax.set_xticklabels(fpv_scens, fontsize=9)
            ax.set_title(period_label, fontsize=10, loc='left')
            ax.grid(True, axis='y', alpha=0.3, zorder=0)
            ax.tick_params(labelsize=9)
            ax.margins(y=0.15)

        axes[0].set_ylabel(label_for(sub), fontsize=10)
        # Single legend on the rightmost panel for Baseline reference
        axes[-1].legend(loc='upper right', fontsize=9, frameon=True)

        plt.tight_layout()
        save_fig(fig, f'{save_name}_{sub}')
        plt.show()


plot_monthly_delta_panels(series_monthly, series_daily, KEY_SUBS,
                          save_name='06_monthly_delta_panels')

In [ ]:
# ── Surface-only (K=K_TOP) perimeter time series ──────────────────────────
def perimeter_mean_surface(ds, perim_da, substance):
    """Perimeter mean at K=K_TOP only, native resolution."""
    if substance not in ds:
        return pd.Series(dtype=float)
    da = ds[substance].where(perim_da).sel(K=K_TOP).mean(['M', 'N'])
    return pd.Series(da.values, index=pd.DatetimeIndex(da.time.values))


def build_series_surface(ds, perim_da, subs):
    out = {}
    for sub in subs:
        s = perimeter_mean_surface(ds, perim_da, sub)
        if not s.empty:
            out[sub] = s
    return pd.DataFrame(out)


# CHANGED: derived fields use the same add_derived() as the layer-mean series (Section 6)


# Use the canonical substance list from series_daily — same fix as before
all_subs = list(series_daily['Baseline'].columns)
DERIVED = {'TotN', 'TotP', 'TotAlgae', 'Chlfa', 'AlgN', 'AlgP'}
direct_subs = [s for s in all_subs if s not in DERIVED]

print(f'Computing surface (K=K_TOP) perimeter series with {len(direct_subs)} '
      f'substances...')
series_daily_surf = {}
for scen in SCEN_ORDER:
    df_s = build_series_surface(runs[scen], perim_da, direct_subs)
    df_s = add_derived(df_s)
    series_daily_surf[scen] = df_s
    print(f'  {scen}: {len(df_s.columns)} substances', flush=True)

# Force daily resample on the surface series — done AFTER the build loop
# completes, in a SEPARATE loop, so all scenarios exist by then.
for scen in SCEN_ORDER:
    series_daily_surf[scen] = series_daily_surf[scen].resample('D').mean()

series_monthly_surf = {scen: df.resample('MS').mean()
                       for scen, df in series_daily_surf.items()}

# Quick sanity check
print('\nSurface vs layer-mean comparison for OXY (Baseline, July mean):')
print(f'  Surface (K={K_TOP}):  '
      f'{series_daily_surf["Baseline"]["OXY"].loc["2023-07-01":"2023-07-31"].mean():.3f} g/m³')
print(f'  {LAYERMEAN_LABEL}: '
      f'{series_daily["Baseline"]["OXY"].loc["2023-07-01":"2023-07-31"].mean():.3f} g/m³')

# Quick verification of ecotype columns
print('\nEcotype columns in series_daily_surf:')
for scen in ['Baseline', 'FPV_02']:
    cols = series_daily_surf[scen].columns
    print(f'  {scen}: '
          f'_E={sum(1 for c in cols if c.endswith("_E"))}, '
          f'_N={sum(1 for c in cols if c.endswith("_N"))}, '
          f'_P={sum(1 for c in cols if c.endswith("_P"))}, '
          f'_F={sum(1 for c in cols if c.endswith("_F"))}')

In [ ]:
# ── Mean Δ bar plots — surface AND depth-averaged paired per scenario ────
# 2×3 panel grid, one panel per substance. Within each panel, 3 scenarios
# (FPV_02/03/04) × 2 bars each (surface + depth-averaged) = 6 bars total.
# Pairing within scenario makes surface-vs-depth flips visually obvious.

KEY_SUBS_DELTA = ['OXY', 'TotAlgae', 'Chlfa', 'NO3', 'PO4', 'NH4']
FPV_SCENS = ['FPV_02', 'FPV_03', 'FPV_04']
FPV_SHORT = {'FPV_02': '02\nno gap', 'FPV_03': '03\n10×10', 'FPV_04': '04\ndonut'}

SURFACE_HATCH = ''      # solid for surface
DEPTH_HATCH   = '///'   # hatched for depth-avg


def plot_paired_delta_bars(surf_dict, depth_dict,
                            surf_label='Surface (K=K_TOP)',
                            depth_label=LAYERMEAN_LABEL,
                            save_name=None):
    """2×3 grid, one panel per substance. Each panel has 3 scenarios × 2
    bars (surface, depth-averaged) showing Δ vs Baseline."""
    subs_present = [s for s in KEY_SUBS_DELTA
                    if s in surf_dict['Baseline'].columns
                    and s in depth_dict['Baseline'].columns]
    if not subs_present:
        print('No matching substances in both surface and depth dicts'); return

    nrows = 2
    ncols = 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 3.0 * nrows),
                              squeeze=False)
    axes = axes.flatten()

    panel_letters = list('abcdefghij')
    bar_w = 0.35

    for ax, sub, letter in zip(axes, subs_present, panel_letters):
        deltas_surf  = []
        deltas_depth = []
        for scen in FPV_SCENS:
            bm_surf  = surf_dict['Baseline'][sub].mean()
            bm_depth = depth_dict['Baseline'][sub].mean()
            d_surf  = (surf_dict[scen][sub].mean() - bm_surf
                       if sub in surf_dict[scen].columns else np.nan)
            d_depth = (depth_dict[scen][sub].mean() - bm_depth
                       if sub in depth_dict[scen].columns else np.nan)
            deltas_surf.append(d_surf)
            deltas_depth.append(d_depth)

        x = np.arange(len(FPV_SCENS))
        colours = [SCEN_COLORS[s] for s in FPV_SCENS]

        bars_s = ax.bar(x - bar_w/2, deltas_surf, bar_w,
                        color=colours, edgecolor='black', lw=0.5,
                        hatch=SURFACE_HATCH,
                        label=surf_label if ax is axes[0] else None)
        bars_d = ax.bar(x + bar_w/2, deltas_depth, bar_w,
                        color=colours, edgecolor='black', lw=0.5,
                        hatch=DEPTH_HATCH, alpha=0.85,
                        label=depth_label if ax is axes[0] else None)

        ax.axhline(0, color='k', lw=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels([FPV_SHORT[s] for s in FPV_SCENS], fontsize=10)
        substance = label_for(sub)
        ax.text(0.98, 0.9, f'({letter}) {substance}', transform=ax.transAxes,
                fontsize=9, ha='right', va='top',
                bbox=dict(facecolor='white', edgecolor='none',
                          pad=2.0, alpha=0.8))
        ax.tick_params(labelsize=9)
        ax.grid(True, axis='y', alpha=0.3, zorder=0)

    for j in range(len(subs_present), len(axes)):
        axes[j].set_visible(False)

    # Y-axis label on every leftmost panel in each row
    for row in range(nrows):
        axes[row * ncols].set_ylabel('Δ(FPV − Baseline)', fontsize=10)

    from matplotlib.patches import Patch
    legend_handles = [
        Patch(facecolor='grey', edgecolor='black', lw=0.5,
              hatch=SURFACE_HATCH, label=surf_label),
        Patch(facecolor='grey', edgecolor='black', lw=0.5,
              hatch=DEPTH_HATCH, alpha=0.85, label=depth_label),
    ]
    fig.legend(handles=legend_handles, loc='upper center',
                ncol=1, fontsize=8, frameon=True,
                bbox_to_anchor=(0.25, 0.875))

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    if save_name:
        save_fig(fig, save_name)
    plt.show()

    # ── Summary table ─────────────────────────────────────────────────────
    print(f'\nMean Δ vs Baseline — Apr-Aug 2023 (surface | layer mean):')
    header = f'  {"Substance":<12} {"Layer":<10}'
    for scen in FPV_SCENS:
        header += f' {"Δ " + scen:>14}'
    print(header)
    print('  ' + '-' * (24 + 15 * len(FPV_SCENS)))
    for sub in subs_present:
        for layer_label, layer_dict in [('Surface', surf_dict),
                                         ('Layer mean', depth_dict)]:
            bm = layer_dict['Baseline'][sub].mean()
            line = f'  {sub:<12} {layer_label:<10}'
            for scen in FPV_SCENS:
                if sub in layer_dict[scen].columns:
                    d = layer_dict[scen][sub].mean() - bm
                    line += f' {d:>+13.4f}'
                else:
                    line += f' {"--":>13}'
            print(line)
        print()


plot_paired_delta_bars(series_daily_surf, series_daily,
                        surf_label='Surface (K=7)',
                        depth_label=LAYERMEAN_LABEL,
                        save_name='06_delta_bars_paired')

In [ ]:
# ── Secchi depth time series — separate figure ──────────────────────────
# Secchi is read from .his (not in .map.nc / series_daily).
# Plotted at CAB monitor surface (k_label=8 = WAQ K_TOP=7) — the only
# field-validated reference point in the reservoir.

def secchi_from_his(his_run, loc_dict=None, loc_name='CAB_averaged'):
    """CHANGED: SecchiDept from .his at CAB_averaged (as in the thesis and Section 8), daily."""
    si = his_run['idx'].get('SecchiDept')
    if si is None or loc_name not in his_run['locations']:
        return pd.Series(dtype=float)
    # Use 'timestamps' key (comprehensive notebook convention) with fallback
    # to 'ts_index' (diagnostic notebook convention)
    time_idx = his_run.get('timestamps', his_run.get('ts_index'))
    if time_idx is None:
        return pd.Series(dtype=float)
    ts = pd.Series(his_run['data'][:, his_run['locations'].index(loc_name), si].astype(float),
                    index=time_idx)
    # Filter physical range (Secchi shouldn't be 0 or negative or absurdly large)
    ts[ts <= 0] = np.nan
    ts[ts > 50] = np.nan
    return ts.resample('D').mean()


# his_runs and cab_locs are assumed to be in scope from earlier cells
# in the comprehensive notebook (the .his loading section).
secchi_series = {}
for scen in SCEN_ORDER:
    if scen in his_runs:
        # CAB monitor index dict — assumes same convention as in your .his loader
        cab_dict = {k: his_runs[scen]['locations'].index(f'CAB ({k})')
                    for k in range(8, 17)
                    if f'CAB ({k})' in his_runs[scen]['locations']}
        secchi_series[scen] = secchi_from_his(his_runs[scen], cab_dict)
    else:
        secchi_series[scen] = pd.Series(dtype=float)

# Plot
fig, ax = plt.subplots(figsize=(6, 3.5))
for scen in SCEN_ORDER:
    if secchi_series[scen].empty:
        print(f'Skipping {scen}: no SecchiDept in .his')
        continue
    ls = '--' if scen == 'Baseline' else '-'
    ax.plot(secchi_series[scen].index, secchi_series[scen].values,
            lw=1.2, color=SCEN_COLORS[scen], ls=ls, label=scen)

ax.set_ylabel('Secchi depth (m)', fontsize=10)
ax.set_xlabel('')
ax.invert_yaxis()   # Conventional: deeper Secchi (more transparent) plotted lower
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.set_xlim(pd.Timestamp('2023-04-01'), pd.Timestamp('2023-08-31'))
ax.legend(fontsize=9, loc='best')

# Add a note about the data source at the top-right inside the plot
# ax.text(0.98, 0.02, 'CAB surface (K_TOP) — from .his',
#         transform=ax.transAxes, fontsize=8,
#         ha='right', va='bottom', color='#444444',
#         bbox=dict(facecolor='white', edgecolor='none', pad=2.0, alpha=0.85))

plt.tight_layout()
save_fig(fig, '07_secchi_depth')
plt.show()

# Print summary
print('\nSecchi depth summary (CAB_averaged, Apr-Aug 2023):')
print(f'  {"Scenario":<10} {"Mean":>10} {"Min":>10} {"Max":>10}')
print('  ' + '-' * 50)
for scen in SCEN_ORDER:
    if not secchi_series[scen].empty:
        s = secchi_series[scen]
        print(f'  {scen:<10} {s.mean():>10.2f} {s.min():>10.2f} {s.max():>10.2f}')

## Section 7 — Daily time series plots (perimeter mean)

In [ ]:
# ── Multi-variable surface vs depth-averaged time series (5 variables) ──
# DO, NO3, NH4, PO4, Si arranged in a 2×3 grid of paired sub-panels.
# Each variable gets a pair (surface on top, depth-avg below).
# The 6th cell of the grid is left empty.

import matplotlib.gridspec as gridspec

VARS_TO_PLOT = ['OXY', 'NO3', 'NH4', 'PO4', 'Si']
n_vars = len(VARS_TO_PLOT)
ncols_outer = 3
nrows_outer = 2   # 2×3 = 6 cells, only 5 used → 1 empty cell at bottom-right

fig = plt.figure(figsize=(9.5, 7))
outer_gs = gridspec.GridSpec(nrows_outer, ncols_outer, figure=fig,
                              wspace=0.30, hspace=0.15)

panel_letters = list('abcdefghij')

for v_idx, sub in enumerate(VARS_TO_PLOT):
    if (sub not in series_daily['Baseline'].columns
            or sub not in series_daily_surf['Baseline'].columns):
        print(f'Skipping {sub}: not in both series'); continue
    row, col = v_idx // ncols_outer, v_idx % ncols_outer
    inner_gs = outer_gs[row, col].subgridspec(2, 1, hspace=0.05)
    ax_surf = fig.add_subplot(inner_gs[0])
    ax_davg = fig.add_subplot(inner_gs[1], sharex=ax_surf)

    for scen in SCEN_ORDER:
        ls = '--' if scen == 'Baseline' else '-'
        if sub in series_daily_surf[scen].columns:
            ax_surf.plot(series_daily_surf[scen].index,
                          series_daily_surf[scen][sub],
                          lw=1.0, color=SCEN_COLORS[scen], ls=ls,
                          label=scen if v_idx == 0 else None)
        if sub in series_daily[scen].columns:
            ax_davg.plot(series_daily[scen].index,
                          series_daily[scen][sub],
                          lw=1.0, color=SCEN_COLORS[scen], ls=ls)

    # Hide tick labels on the top sub-panel x-axis
    plt.setp(ax_surf.get_xticklabels(), visible=False)
    ax_surf.tick_params(labelsize=7)
    ax_surf.grid(True, alpha=0.3)
    ax_davg.tick_params(labelsize=7)
    ax_davg.grid(True, alpha=0.3)
    ax_davg.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
    ax_davg.xaxis.set_major_locator(mdates.MonthLocator())

    # No per-axis y-label — add a centered annotation between the pair
    ax_surf.set_ylabel('')
    ax_davg.set_ylabel('')

    # Center y-label between the two sub-panels
    pos_surf = ax_surf.get_position()
    pos_davg = ax_davg.get_position()
    y_centre = (pos_davg.y0 + pos_surf.y1) / 2
    x_label  = pos_surf.x0 - 0.04
    fig.text(x_label, y_centre, label_for(sub),
             rotation='vertical', ha='center', va='center', fontsize=9)

    # Surface / Depth-avg annotation INSIDE each sub-panel (top-right)
    ax_surf.text(0.98, 0.95, 'Surface', transform=ax_surf.transAxes,
                  fontsize=8, ha='right', va='top', color='#444444',
                  bbox=dict(facecolor='white', edgecolor='none',
                            pad=2.0, alpha=0.85))
    ax_davg.text(0.98, 0.95, 'Layer mean', transform=ax_davg.transAxes,
                  fontsize=8, ha='right', va='top', color='#444444',
                  bbox=dict(facecolor='white', edgecolor='none',
                            pad=2.0, alpha=0.85))

    # Panel letters: (a.i) inside surface, (a.ii) inside depth-avg
    letter = panel_letters[v_idx]
    for ax_, suffix in [(ax_surf, 'i'), (ax_davg, 'ii')]:
        ax_.text(0.02, 0.95, f'({letter}.{suffix})',
                  transform=ax_.transAxes, fontsize=9,
                  ha='left', va='top',
                  bbox=dict(facecolor='white', edgecolor='none',
                            pad=2.0, alpha=0.85))

fig.legend(loc='upper center', ncol=1, fontsize=8,
            bbox_to_anchor=(0.7, 0.45), frameon=True)
plt.tight_layout(rect=[0, 0, 1, 0.97])
save_fig(fig, '07_surface_vs_depthavg_DO_nutrients')
plt.show()

In [ ]:
# ── DO time series: surface vs depth-averaged, all scenarios ─────────────
# Two-panel figure with shared x-axis. Panel (a) shows surface DO (K=K_TOP),
# panel (b) shows depth-averaged DO (K=7..15). Both are perimeter means
# over the FPV_02 array footprint.

if 'OXY' in series_daily['Baseline'].columns:
    fig, axes = plt.subplots(2, 1, figsize=(6, 5), sharex=True)

    panel_data = [
        (axes[0], series_daily_surf, 'Surface (K=7)', '(a)'),
        (axes[1], series_daily,      LAYERMEAN_LABEL, '(b)'),
    ]

    for ax, series_dict, layer_label, letter in panel_data:
        for scen in SCEN_ORDER:
            if 'OXY' not in series_dict[scen].columns:
                continue
            ls = '--' if scen == 'Baseline' else '-'
            ax.plot(series_dict[scen].index, series_dict[scen]['OXY'],
                    lw=1.4, color=SCEN_COLORS[scen],
                    label=scen, ls=ls, alpha=0.95)

        ax.set_ylabel(label_for('OXY'), fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelsize=9)
        add_panel_letter(ax, f'{letter} {layer_label}',
                         x=0.01, y=0.97, fontsize=9)

    axes[0].legend(fontsize=9, ncol=2, loc='upper right')
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
    axes[-1].xaxis.set_major_locator(mdates.MonthLocator())

    plt.tight_layout()
    save_fig(fig, '07_DO_surface_vs_depthavg')
    plt.show()

    # Print a small summary table
    print('\nDO summary (FPV perimeter mean, Apr-Aug 2023):')
    print(f'  {"Scenario":<10} {"Surface mean":>14} {"Layer mean":>16} '
          f'{"Δ surface":>14} {"Δ layer mean":>14}')
    print('  ' + '-' * 75)
    base_surf = series_daily_surf['Baseline']['OXY'].mean()
    base_davg = series_daily['Baseline']['OXY'].mean()
    for scen in SCEN_ORDER:
        v_surf = series_daily_surf[scen]['OXY'].mean()
        v_davg = series_daily[scen]['OXY'].mean()
        d_surf = v_surf - base_surf if scen != 'Baseline' else 0.0
        d_davg = v_davg - base_davg if scen != 'Baseline' else 0.0
        d_surf_str = '' if scen == 'Baseline' else f'{d_surf:>+13.4f}'
        d_davg_str = '' if scen == 'Baseline' else f'{d_davg:>+13.4f}'
        print(f'  {scen:<10} {v_surf:>14.4f} {v_davg:>16.4f} '
              f'{d_surf_str:>14} {d_davg_str:>14}')
else:
    print('OXY not in series_daily — skipping DO surface vs depth-avg plot.')

In [ ]:
# ── Multi-variable surface vs depth-averaged time series ─────────────────
# Each variable gets a pair of stacked sub-panels (surface on top,
# depth-avg below). 6 variables arranged in a 2×3 grid of pairs.

import matplotlib.gridspec as gridspec

VARS_TO_PLOT = ['OXY', 'NH4', 'NO3', 'PO4', 'TotN', 'TotP']
n_vars = len(VARS_TO_PLOT)
ncols_outer = 3
nrows_outer = int(np.ceil(n_vars / ncols_outer))

fig = plt.figure(figsize=(10.5, 7))
outer_gs = gridspec.GridSpec(nrows_outer, ncols_outer, figure=fig,
                              wspace=0.30, hspace=0.15)

panel_letters = list('abcdefghij')

for v_idx, sub in enumerate(VARS_TO_PLOT):
    if (sub not in series_daily['Baseline'].columns
            or sub not in series_daily_surf['Baseline'].columns):
        print(f'Skipping {sub}: not in both series'); continue

    row, col = v_idx // ncols_outer, v_idx % ncols_outer
    inner_gs = outer_gs[row, col].subgridspec(2, 1, hspace=0.05)
    ax_surf = fig.add_subplot(inner_gs[0])
    ax_davg = fig.add_subplot(inner_gs[1], sharex=ax_surf)

    for scen in SCEN_ORDER:
        ls = '--' if scen == 'Baseline' else '-'
        if sub in series_daily_surf[scen].columns:
            ax_surf.plot(series_daily_surf[scen].index,
                          series_daily_surf[scen][sub],
                          lw=1.0, color=SCEN_COLORS[scen], ls=ls,
                          label=scen if v_idx == 0 else None)
        if sub in series_daily[scen].columns:
            ax_davg.plot(series_daily[scen].index,
                          series_daily[scen][sub],
                          lw=1.0, color=SCEN_COLORS[scen], ls=ls)

    # Hide tick labels on the top sub-panel x-axis
    plt.setp(ax_surf.get_xticklabels(), visible=False)
    ax_surf.tick_params(labelsize=7)
    ax_surf.grid(True, alpha=0.3)
    ax_davg.tick_params(labelsize=7)
    ax_davg.grid(True, alpha=0.3)
    ax_davg.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
    ax_davg.xaxis.set_major_locator(mdates.MonthLocator())

    # No per-axis y-label — add a centered annotation between the pair instead
    ax_surf.set_ylabel('')
    ax_davg.set_ylabel('')

    # Center y-label between the two sub-panels
    # Compute the vertical centre between the two axes' positions
    pos_surf = ax_surf.get_position()
    pos_davg = ax_davg.get_position()
    y_centre = (pos_davg.y0 + pos_surf.y1) / 2
    x_label  = pos_surf.x0 - 0.04   # 4% to the left of panel edge
    fig.text(x_label, y_centre, label_for(sub),
             rotation='vertical', ha='center', va='center', fontsize=9)

    # Surface / Depth-avg annotation INSIDE each sub-panel (top-right)
    ax_surf.text(0.98, 0.95, 'Surface', transform=ax_surf.transAxes,
                  fontsize=8,
                  ha='right', va='top', color='#444444',
                  bbox=dict(facecolor='white', edgecolor='none',
                            pad=2.0, alpha=0.85))
    ax_davg.text(0.98, 0.95, 'Layer mean', transform=ax_davg.transAxes,
                  fontsize=8, 
                  ha='right', va='top', color='#444444',
                  bbox=dict(facecolor='white', edgecolor='none',
                            pad=2.0, alpha=0.85))

    # Panel letters: (a.i) inside surface, (a.ii) inside depth-avg
    letter = panel_letters[v_idx]
    for ax_, suffix in [(ax_surf, 'i'), (ax_davg, 'ii')]:
        ax_.text(0.02, 0.95, f'({letter}.{suffix})',
                  transform=ax_.transAxes,
                  fontsize=9,
                  ha='left', va='top',
                  bbox=dict(facecolor='white', edgecolor='none',
                            pad=2.0, alpha=0.85))

fig.legend(loc='upper center', ncol=1, fontsize=8,
            bbox_to_anchor=(0.95, 0.885), frameon=True)
plt.tight_layout(rect=[0, 0, 1, 0.97])
save_fig(fig, '07_surface_vs_depthavg_multivar')
plt.show()

In [ ]:
# ── Multi-variable surface vs depth-averaged time series — phyto & particulates ──
# Each variable gets a pair of stacked sub-panels (surface on top,
# depth-avg below). 6 variables arranged in a 2×3 grid of pairs.

import matplotlib.gridspec as gridspec

VARS_TO_PLOT = ['TotAlgae', 'Chlfa', 'POC1', 'PON1', 'POP1', 'Si']
n_vars = len(VARS_TO_PLOT)
ncols_outer = 3
nrows_outer = int(np.ceil(n_vars / ncols_outer))

fig = plt.figure(figsize=(11, 7))
outer_gs = gridspec.GridSpec(nrows_outer, ncols_outer, figure=fig,
                              wspace=0.30, hspace=0.15)

panel_letters = list('abcdefghij')

for v_idx, sub in enumerate(VARS_TO_PLOT):
    if (sub not in series_daily['Baseline'].columns
            or sub not in series_daily_surf['Baseline'].columns):
        print(f'Skipping {sub}: not in both series'); continue

    row, col = v_idx // ncols_outer, v_idx % ncols_outer
    inner_gs = outer_gs[row, col].subgridspec(2, 1, hspace=0.05)
    ax_surf = fig.add_subplot(inner_gs[0])
    ax_davg = fig.add_subplot(inner_gs[1], sharex=ax_surf)

    for scen in SCEN_ORDER:
        ls = '--' if scen == 'Baseline' else '-'
        if sub in series_daily_surf[scen].columns:
            ax_surf.plot(series_daily_surf[scen].index,
                          series_daily_surf[scen][sub],
                          lw=1.0, color=SCEN_COLORS[scen], ls=ls,
                          label=scen if v_idx == 0 else None)
        if sub in series_daily[scen].columns:
            ax_davg.plot(series_daily[scen].index,
                          series_daily[scen][sub],
                          lw=1.0, color=SCEN_COLORS[scen], ls=ls)

    plt.setp(ax_surf.get_xticklabels(), visible=False)
    ax_surf.tick_params(labelsize=7)
    ax_surf.grid(True, alpha=0.3)
    ax_davg.tick_params(labelsize=7)
    ax_davg.grid(True, alpha=0.3)
    ax_davg.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
    ax_davg.xaxis.set_major_locator(mdates.MonthLocator())

    ax_surf.set_ylabel('')
    ax_davg.set_ylabel('')
    pos_surf = ax_surf.get_position()
    pos_davg = ax_davg.get_position()
    y_centre = (pos_davg.y0 + pos_surf.y1) / 2
    x_label  = pos_surf.x0 - 0.05
    fig.text(x_label, y_centre, label_for(sub),
             rotation='vertical', ha='center', va='center', fontsize=9)

    ax_surf.text(0.98, 0.95, 'Surface', transform=ax_surf.transAxes,
                  fontsize=8,
                  ha='right', va='top', color='#444444',
                  bbox=dict(facecolor='white', edgecolor='none',
                            pad=2.0, alpha=0.85))
    ax_davg.text(0.98, 0.95, 'Layer mean', transform=ax_davg.transAxes,
                  fontsize=8,
                  ha='right', va='top', color='#444444',
                  bbox=dict(facecolor='white', edgecolor='none',
                            pad=2.0, alpha=0.85))

    letter = panel_letters[v_idx]
    for ax_, suffix in [(ax_surf, 'i'), (ax_davg, 'ii')]:
        ax_.text(0.02, 0.95, f'({letter}.{suffix})',
                  transform=ax_.transAxes,
                  fontsize=9,
                  ha='left', va='top',
                  bbox=dict(facecolor='white', edgecolor='none',
                            pad=2.0, alpha=0.85))

fig.legend(loc='upper center', ncol=1, fontsize=8,
            bbox_to_anchor=(0.95, 0.885), frameon=True)

plt.tight_layout(rect=[0, 0, 1, 0.97])
save_fig(fig, '07_surface_vs_depthavg_phyto')
plt.show()

## Section 8 — Secchi depth (`.his` at CAB_averaged)

In [ ]:
if CAB_AVERAGED_IDX is None or 'SecchiDept' not in his_runs['Baseline']['idx']:
    print('Skipping Secchi: CAB_averaged or SecchiDept not in .his.')
else:
    secchi_si = his_runs['Baseline']['idx']['SecchiDept']
    secchi = {}
    for scen, r in his_runs.items():
        ts = r['data'][:, CAB_AVERAGED_IDX, secchi_si].astype(float)
        ts[ts <= 0] = np.nan
        secchi[scen] = pd.Series(ts, index=r['timestamps'])

    fig, axes = plt.subplots(2, 1, figsize=(7, 4.5), sharex=True,
                             gridspec_kw={'height_ratios': [2, 1]})
    for scen in SCEN_ORDER:
        ls = '--' if scen == 'Baseline' else '-'
        axes[0].plot(secchi[scen].index, secchi[scen].values,
                     lw=1.3, color=SCEN_COLORS[scen], label=scen, ls=ls)
    axes[0].set_ylabel('Secchi depth (m)', fontsize=10)
    axes[0].invert_yaxis()
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(fontsize=9, ncol=4, loc='lower left')
    add_panel_letter(axes[0], '(a)', fontsize=10)

    base = secchi['Baseline']
    for scen in ['FPV_02', 'FPV_03', 'FPV_04']:
        pct = (secchi[scen] - base) / base * 100
        axes[1].plot(pct.index, pct.values, lw=1.2,
                     color=SCEN_COLORS[scen], label=scen)
    axes[1].axhline(0, color='k', lw=0.5, alpha=0.5)
    axes[1].set_ylabel('Secchi depth deviation\nfrom baseline run (%)', fontsize=10)
    axes[1].grid(True, alpha=0.3)
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
    axes[1].xaxis.set_major_locator(mdates.MonthLocator())
    add_panel_letter(axes[1], '(b)', fontsize=10)

    plt.tight_layout()
    save_fig(fig, '08_secchi_cab_averaged')
    plt.show()

    # Seasonal summary
    print('\nSecchi seasonal means at CAB_averaged:')
    print(f'  {"Scenario":<10} {"Season":<22} {"Mean (m)":>10} {"Δ Base (%)":>14}')
    print('  ' + '-' * 60)
    for scen in SCEN_ORDER:
        for season, (t0, t1) in SEASONS.items():
            v = secchi[scen].loc[t0:t1].mean()
            b = secchi['Baseline'].loc[t0:t1].mean()
            pct = (v - b) / b * 100 if scen != 'Baseline' else 0.0
            tag = '' if scen == 'Baseline' else f'{pct:>+13.2f}%'
            print(f'  {scen:<10} {season:<22} {v:>10.3f} {tag:>14}')

## Section 9 — Species composition

Stacked area (absolute + %), per-species deltas, and monthly bar plots for
each species group across all scenarios.

In [ ]:
# ── Combined figure: absolute biomass + % composition, all 4 scenarios ──
# 4 rows × 2 cols layout. Each row = one scenario.
#   Left column:  absolute biomass stacked (gC/m³) — shared y across column
#   Right column: % biomass composition (0-100%) — shared y across column
# Data source: series_daily (perimeter mean, depth-avg) from comprehensive notebook.

import matplotlib.gridspec as gridspec


SPECIES_ECOTYPES_BAR = {
    'Diatoms':       ['FDIATOMS_E', 'FDIATOMS_N', 'FDIATOMS_P'],
    'Flagellates':   ['FFLAGELA'],
    'Greens':        ['GREENS_E',   'GREENS_N',   'GREENS_P'],
    'Cyanobacteria': ['MICROCYS_E', 'MICROCYS_N', 'MICROCYS_P'],
}
GROUP_COLOURS_BAR = {
    'Diatoms':       '#ff7f0e',
    'Flagellates':   '#1f77b4',
    'Greens':        '#2ca02c',
    'Cyanobacteria': '#9467bd',
}
GROUPS_ORDER = ['Diatoms', 'Flagellates', 'Greens', 'Cyanobacteria']


def group_biomass_per_scenario(scen):
    """Sum biomass per group for one scenario from series_daily."""
    df = series_daily[scen]
    out = {}
    for grp, ecotypes in SPECIES_ECOTYPES_BAR.items():
        present = [s for s in ecotypes if s in df.columns]
        if present:
            out[grp] = df[present].sum(axis=1)
        else:
            out[grp] = pd.Series(0.0, index=df.index)
    return pd.DataFrame(out)[GROUPS_ORDER]


# Build per-scenario group biomass dicts
group_biomass = {scen: group_biomass_per_scenario(scen) for scen in SCEN_ORDER}
group_percent = {scen: df.div(df.sum(axis=1), axis=0) * 100
                 for scen, df in group_biomass.items()}


# Determine shared y-limit for absolute column (so FPV_02's larger peak shows)
abs_ymax = max(df.sum(axis=1).max() for df in group_biomass.values()) * 1.05


# ── Build figure: 4 rows × 2 cols ───────────────────────────────────────
fig = plt.figure(figsize=(8.5, 6))
gs = gridspec.GridSpec(4, 2, figure=fig,
                       left=0.06, right=0.97, top=0.95, bottom=0.08,
                       hspace=0.10, wspace=0.15,
                       width_ratios=[1, 1])

panel_letters = list('abcdefgh')
axes_abs = []
axes_pct = []

for i, scen in enumerate(SCEN_ORDER):
    # Absolute biomass (left column)
    sharex_abs = None if i == 0 else axes_abs[0]
    sharey_abs = None if i == 0 else axes_abs[0]
    ax_abs = fig.add_subplot(gs[i, 0], sharex=sharex_abs, sharey=sharey_abs)
    axes_abs.append(ax_abs)

    df = group_biomass[scen]
    colours = [GROUP_COLOURS_BAR[g] for g in df.columns]
    ax_abs.stackplot(df.index, *[df[g].values for g in df.columns],
                     labels=df.columns.tolist(), colors=colours, alpha=0.85)
    ax_abs.set_ylabel('Biomass (gC/m³)', fontsize=9)
    ax_abs.set_ylim(0, abs_ymax)
    ax_abs.grid(True, alpha=0.3)
    ax_abs.tick_params(labelsize=8)

    letter_abs = panel_letters[i * 2]
    ax_abs.text(0.02, 0.97, f'({letter_abs}) {scen}',
                transform=ax_abs.transAxes, fontsize=9, ha='left', va='top',
                bbox=dict(facecolor='white', edgecolor='none',
                          pad=2.0, alpha=0.85))

    # % composition (right column)
    sharex_pct = None if i == 0 else axes_pct[0]
    sharey_pct = None if i == 0 else axes_pct[0]
    ax_pct = fig.add_subplot(gs[i, 1], sharex=sharex_pct, sharey=sharey_pct)
    axes_pct.append(ax_pct)

    dfp = group_percent[scen]
    ax_pct.stackplot(dfp.index, *[dfp[g].values for g in dfp.columns],
                     labels=dfp.columns.tolist(), colors=colours, alpha=0.85)
    ax_pct.set_ylabel('% biomass', fontsize=9)
    ax_pct.set_ylim(0, 100)
    ax_pct.grid(True, alpha=0.3)
    ax_pct.tick_params(labelsize=8)

    letter_pct = panel_letters[i * 2 + 1]
    ax_pct.text(0.02, 0.97, f'({letter_pct}) {scen}',
                transform=ax_pct.transAxes, fontsize=9, ha='left', va='top',
                bbox=dict(facecolor='white', edgecolor='none',
                          pad=2.0, alpha=0.85))

# Hide x-tick labels on non-bottom rows
for ax in axes_abs[:-1] + axes_pct[:-1]:
    plt.setp(ax.get_xticklabels(), visible=False)

# X-axis formatting on bottom row
for ax in [axes_abs[-1], axes_pct[-1]]:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.set_xlim(pd.Timestamp('2023-04-01'), pd.Timestamp('2023-08-31'))

# Single legend at top-right of panel (b)
axes_abs[0].legend(fontsize=8, loc='upper right', framealpha=0.95)

save_fig(fig, '09_combined_biomass_absolute_and_percent')
plt.show()

In [ ]:
# ── 9.4 — Monthly species composition bar plots (improved visibility) ──────
groups_monthly = {}
for scen in SCEN_ORDER:
    gm = {}
    for grp_name, ecotypes in SPECIES_GROUPS.items():
        present = [s for s in ecotypes if s in series_daily[scen].columns]
        if present:
            gm[grp_name] = series_daily[scen][present].sum(axis=1).resample('MS').mean()
        else:
            gm[grp_name] = pd.Series(0, index=series_daily[scen].index).resample('MS').mean()
    groups_monthly[scen] = pd.DataFrame(gm)

months = groups_monthly['Baseline'].index
month_labels = [m.strftime('%b\n%Y') for m in months]
n_months = len(months)
n_scen = len(SCEN_ORDER)

SCEN_SHORT = {'Baseline': 'B', 'FPV_02': '02', 'FPV_03': '03', 'FPV_04': '04'}

bar_w = 0.20
fig_w = max(5, n_months * 1.3)


def _draw_grouped_stacked(ax, value_func, ylabel, ylim_top=None):
    """Common drawing routine for absolute and % composition bar charts."""
    x = np.arange(n_months)
    for xi in x[:-1]:
        ax.axvline(xi + 0.5, color='#cccccc', lw=0.5, ls=':', zorder=0)

    for j, scen in enumerate(SCEN_ORDER):
        offset = (j - (n_scen - 1) / 2) * bar_w
        bottom = np.zeros(n_months)
        for grp_name in SPECIES_GROUPS:
            vals = value_func(scen, grp_name)
            ax.bar(x + offset, vals, bar_w,
                   bottom=bottom, color=GROUP_COLORS[grp_name],
                   edgecolor='black', lw=0.4, alpha=0.92,
                   label=grp_name if j == 0 else None)
            bottom = bottom + np.nan_to_num(vals)
        for xi, h in zip(x, bottom):
            ax.annotate(SCEN_SHORT[scen],
                        xy=(xi + offset, h),
                        xytext=(0, 3), textcoords='offset points',
                        ha='center', va='bottom', fontsize=8,
                        color='k', 
                        clip_on=False)

    ax.set_xticks(x)
    ax.set_xticklabels(month_labels, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(True, axis='y', alpha=0.3, zorder=0)
    ax.tick_params(labelsize=9)
    if ylim_top is not None:
        ax.set_ylim(0, ylim_top)
    else:
        cur_top = ax.get_ylim()[1]
        ax.set_ylim(0, cur_top * 1.20)


# ── Single figure with two stacked panels ────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(fig_w, 5), sharex=True)

# (a) Absolute monthly biomass
_draw_grouped_stacked(
    axes[0],
    value_func=lambda scen, grp: groups_monthly[scen][grp].values,
    ylabel='Biomass (gC/m³)',
)
add_panel_letter(axes[0], '(a)', x=0.02, y=0.95, fontsize=10)

grp_handles = [Patch(facecolor=GROUP_COLORS[g], edgecolor='black', lw=0.4,
                     label=g) for g in SPECIES_GROUPS]
axes[0].legend(handles=grp_handles, fontsize=9, ncol=2,
               loc='upper right', frameon=True)


# (b) Relative monthly composition
def _pct_value(scen, grp):
    total = sum(groups_monthly[scen][g].values for g in SPECIES_GROUPS)
    safe = np.where(total > 0, total, np.nan)
    return groups_monthly[scen][grp].values / safe * 100

_draw_grouped_stacked(
    axes[1],
    value_func=_pct_value,
    ylabel='% biomass',
    ylim_top=115,
)
add_panel_letter(axes[1], '(b)', x=0.02, y=0.95, fontsize=9)

# axes[1].legend(handles=grp_handles, fontsize=9, ncol=4,
#                loc='lower left', frameon=True)

# fig.suptitle('Monthly species composition — all scenarios',
#              fontsize=12, y=0.995)

plt.tight_layout()
save_fig(fig, '09_4_monthly_composition_combined')
plt.show()

In [ ]:
# ── Section 9 — Biomass-weighted limitation (E/N/P ecotypes), 4-scenario ────
# Sum E-, N-, P-adapted ecotypes across Diatoms, Greens, Cyanobacteria;
# Flagellates excluded (no E/N/P split). APHANFIX_F tracked separately.
# Averaging: FPV_02 perimeter mean across active layers K=7..15.

ALL_E = ['FDIATOMS_E', 'GREENS_E',
         'MICROCYS_E', 'APHANFIX_E', 'OSCILAT_E', 'ANABAENA_E']
ALL_N = ['FDIATOMS_N', 'GREENS_N',
         'MICROCYS_N', 'APHANFIX_N', 'OSCILAT_N', 'ANABAENA_N']
ALL_P = ['FDIATOMS_P', 'GREENS_P',
         'MICROCYS_P', 'APHANFIX_P', 'OSCILAT_P', 'ANABAENA_P']
ALL_F = ['APHANFIX_F']

ECOTYPE_COLOURS = {
    'E': '#2ca02c',   # green — energy/light-adapted
    'N': '#1f77b4',   # blue  — N-adapted
    'P': '#d62728',   # red   — P-adapted
    'F': '#9467bd',   # purple — N-fixing free
}
ECOTYPE_LABELS = {
    'E': 'E (energy/light-adapted)',
    'N': 'N (N-adapted)',
    'P': 'P (P-adapted)',
    'F': 'F (N-fixing free)',
}


def perimeter_ecotype_series(ds, perim_da, ecotypes,
                              k_slice=slice(K_TOP, K_BOT)):
    """Sum the listed ecotype substances at each timestep, taking the
    FPV_02-perimeter mean across active layers."""
    total = None
    for sp in ecotypes:
        if sp not in ds.data_vars:
            continue
        da = ds[sp].where(perim_da).sel(K=k_slice).mean(['M', 'N', 'K'])
        s = pd.Series(da.values, index=pd.DatetimeIndex(da.time.values))
        s = s.resample('D').mean()
        total = s if total is None else total.add(s, fill_value=0)
    if total is None:
        return pd.Series(dtype=float)
    return total


def build_perimeter_ecotype_df(ds, perim_da):
    """DataFrame with columns E, N, P, F — perimeter-mean carbon biomass
    by ecotype, summed across all groups, integrated over K=7..15."""
    cols = {}
    cols['E'] = perimeter_ecotype_series(ds, perim_da, ALL_E)
    cols['N'] = perimeter_ecotype_series(ds, perim_da, ALL_N)
    cols['P'] = perimeter_ecotype_series(ds, perim_da, ALL_P)
    cols['F'] = perimeter_ecotype_series(ds, perim_da, ALL_F)
    # Ensure datetime index even when some cols are empty
    ref_idx = None
    for s in cols.values():
        if not s.empty:
            ref_idx = s.index
            break
    if ref_idx is None:
        return pd.DataFrame()
    for k, s in cols.items():
        if s.empty:
            cols[k] = pd.Series(0.0, index=ref_idx)
    df = pd.concat(cols, axis=1).fillna(0)
    df.index = pd.DatetimeIndex(df.index)
    return df


# Build per-scenario DataFrames
ecot_by_scen = {scen: build_perimeter_ecotype_df(runs[scen], perim_da)
                for scen in SCEN_ORDER}

# Decide whether to show the F ecotype anywhere
f_max = max(ecot_by_scen[scen]['F'].max() for scen in SCEN_ORDER)
plot_ecotypes = ['E', 'N', 'P'] + (['F'] if f_max > 1e-6 else [])
print(f'F-ecotype max biomass across scenarios: {f_max:.6g} gC/m³  '
      f'({"included" if f_max > 1e-6 else "skipped — effectively zero"})')


# ── 4-panel grid: one stacked area per scenario ────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(8, 5.5), sharex=True, sharey=True)
axes_flat = axes.flatten()

panel_labels = ['(a)', '(b)', '(c)', '(d)']
for ax, scen, plabel in zip(axes_flat, SCEN_ORDER, panel_labels):
    df = ecot_by_scen[scen]
    stack = [df[e].values for e in plot_ecotypes]
    colours = [ECOTYPE_COLOURS[e] for e in plot_ecotypes]
    labels = [ECOTYPE_LABELS[e] for e in plot_ecotypes]
    ax.stackplot(df.index, stack, labels=labels, colors=colours, alpha=0.85)
    ax.set_title(f'{plabel} {scen}',
                 fontsize=9, loc='left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.tick_params(labelsize=8)
    ax.grid(True, alpha=0.3)

axes_flat[0].legend(fontsize=7, loc='upper left')

axes_flat[0].set_ylabel('Biomass (gC/m³)', fontsize=9)
axes_flat[2].set_ylabel('Biomass (gC/m³)', fontsize=9)

# fig.suptitle('Biomass-weighted limitation (E/N/P ecotypes) — '
#              'FPV_02 perimeter mean (K=7..15)',
#              fontsize=11, y=1.00)

# fig.text(0.5, -0.005,
#          'Interpretation: dominance of an ecotype indicates which limitation '
#          'regime the standing biomass evolved under.\n'
#          'P-adapted dominance → cells holding high internal P → externally N-limited. '
#          'This is a biomass signature of historical conditions, not the current rate-limit.',
#          ha='center', fontsize=8, color='grey')

plt.tight_layout()
save_fig(fig, '09_ecotype_biomass_all_scenarios')
plt.show()


# ── Monthly mean table ─────────────────────────────────────────────────
print('\nEcotype perimeter-mean biomass — monthly means (gC/m³):')
hdr = f'  {"Scenario":<10} {"Month":<10}'
for e in plot_ecotypes:
    hdr += f'  {ECOTYPE_LABELS[e][:5]:>8}'
print(hdr)
print('  ' + '-' * (24 + 10 * len(plot_ecotypes)))
for scen in SCEN_ORDER:
    monthly = ecot_by_scen[scen].resample('MS').mean()
    for month, row in monthly.iterrows():
        line = f'  {scen:<10} {month.strftime("%Y-%m"):<10}'
        for e in plot_ecotypes:
            line += f'  {row[e]:>8.4f}'
        print(line)
    print()

In [ ]:
# ── Section 9 — BLOOM limitation flags at CAB(8), 4-scenario comparison ──── (CHANGED)
# Limit nit / Limit pho / Limit sil / Limit e / Limit gro / Limit mor are written to the .his only
# (point monitor CAB(8), not a perimeter mean). They are ON/OFF FLAGS: 1 = that constraint binds
# BLOOM's solution at that time step, 0 = it does not. They are not 0-1 efficiency factors.
# The daily mean of a flag = fraction of the day's records in which the constraint binds;
# the monthly mean = fraction of time it binds that month.

LIMIT_VARS = {
    'Limit nit': ('N limitation',            '#1f77b4'),
    'Limit pho': ('P limitation',            '#d62728'),
    'Limit sil': ('Si limitation',           '#8c564b'),
    'Limit e':   ('Light/energy limitation', '#2ca02c'),
    'Limit gro': ('Max. growth rate',        '#9467bd'),
    'Limit mor': ('Mortality',               '#7f7f7f'),
}


def his_limit_series_4scen(his_run, var_name, k_label=8):
    """Daily fraction of time the flag var_name is ON at CAB(k_label)."""
    loc = f'CAB ({k_label})'
    if loc not in his_run['locations'] or var_name not in his_run['idx']:
        return pd.Series(dtype=float)
    ts = pd.Series(his_run['data'][:, his_run['locations'].index(loc), his_run['idx'][var_name]].astype(float),
                   index=pd.DatetimeIndex(his_run['timestamps']))
    return ts.resample('D').mean()


limit_by_scen = {}
for scen in SCEN_ORDER:
    cols = {v: his_limit_series_4scen(his_runs[scen], v, k_label=8) for v in LIMIT_VARS}
    limit_by_scen[scen] = pd.DataFrame({v: ts for v, ts in cols.items() if not ts.empty})

present = [v for v in LIMIT_VARS if all(v in df.columns for df in limit_by_scen.values())]
print(f'Limitation flags present in .his (all scenarios): {present}')

# ── Monthly fraction of time each flag binds: one bar panel per scenario ──
fig, axes = plt.subplots(2, 2, figsize=(9, 6), sharex=True, sharey=True)
for ax, scen, plabel in zip(axes.flatten(), SCEN_ORDER, ['(a)', '(b)', '(c)', '(d)']):
    mo = limit_by_scen[scen][present].resample('MS').mean()
    x = np.arange(len(mo)); w = 0.8 / max(len(present), 1)
    for i, v in enumerate(present):
        label, colour = LIMIT_VARS[v]
        ax.bar(x + (i - (len(present) - 1) / 2) * w, mo[v].values, w, color=colour,
               edgecolor='black', lw=0.3, label=label)
    ax.set_xticks(x, labels=[m.strftime('%b\n%Y') for m in mo.index], fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.grid(True, axis='y', alpha=0.3)
    add_panel_letter(ax, f'{plabel} {scen}', fontsize=9)
axes[0, 0].set_ylabel('Fraction of time binding', fontsize=9)
axes[1, 0].set_ylabel('Fraction of time binding', fontsize=9)
h, l = axes[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc='lower center', ncol=len(l), fontsize=8, frameon=False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
save_fig(fig, '09_limit_flags_all_scenarios')
plt.show()

# ── Tables ──────────────────────────────────────────────────────────────
print('\nFraction of time each limitation binds at CAB(8), Apr-Aug 2023:')
print(pd.DataFrame({scen: limit_by_scen[scen][present].mean() for scen in SCEN_ORDER})
        .rename(index={v: LIMIT_VARS[v][0] for v in present}).round(2).to_string())
print('\nMonthly fraction of time binding at CAB(8):')
for scen in SCEN_ORDER:
    mo = limit_by_scen[scen][present].resample('MS').mean()
    mo.index = mo.index.strftime('%Y-%m')
    print(f'  {scen}')
    print(mo.rename(columns={v: LIMIT_VARS[v][0] for v in present}).round(2).to_string(), '\n')

In [ ]:
# ── Ecotype carbon biomass — Baseline vs FPV_02, FPV perimeter mean ─────
# Sum carbon biomass across all algae species by ecotype letter (_E/_N/_P/_F).
# Same approach as Section 14 of the diagnostic notebook, but perimeter-
# averaged across the FPV array footprint instead of CAB-point.

ECOTYPE_SUFFIXES = {
    'E (light-adapted)':       '_E',
    'N (N-adapted)':           '_N',
    'P (P-adapted)':           '_P',
    'F (N-fixing)':            '_F',
}
ECOTYPE_COLOURS = {
    'E (light-adapted)': '#2ca02c',   # green
    'N (N-adapted)':     '#1f77b4',   # blue
    'P (P-adapted)':     '#d62728',   # red
    'F (N-fixing)':      '#9467bd',   # purple
}

# Flagellates have no suffix — treat as E by convention (light-adapted only)
EXTRA_E_SPECIES = ['FFLAGELA']


def sum_ecotype_perim(series_dict, scen, suffix):
    """Sum all columns in series_daily[scen] whose name ends with `suffix`."""
    cols = [c for c in series_dict[scen].columns if c.endswith(suffix)]
    if suffix == '_E':
        cols += [c for c in EXTRA_E_SPECIES if c in series_dict[scen].columns]
    if not cols:
        return pd.Series(dtype=float)
    return series_dict[scen][cols].sum(axis=1)


# Build ecotype biomass per scenario
ecotype_biomass = {}
for scen in ['Baseline', 'FPV_02']:
    ecotype_biomass[scen] = {}
    for label, suffix in ECOTYPE_SUFFIXES.items():
        ecotype_biomass[scen][label] = sum_ecotype_perim(
            series_daily_surf, scen, suffix)


# ── Plot: 3 panels, sharing x-axis ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
# Share y between panels (a) and (b) — they show absolute biomass on the
# same units, so direct visual comparison benefits from a common scale.
axes[1].sharey(axes[0])
plt.setp(axes[1].get_yticklabels(), visible=False)

# Panel (a) Baseline — stacked
ax = axes[0]
stack_b = []
labels = []
for label in ECOTYPE_SUFFIXES.keys():
    series = ecotype_biomass['Baseline'][label]
    if series.empty:
        continue
    stack_b.append(series.values)
    labels.append(label)
times_b = ecotype_biomass['Baseline'][list(ECOTYPE_SUFFIXES.keys())[0]].index
colours_b = [ECOTYPE_COLOURS[lab] for lab in labels]
ax.stackplot(times_b, *stack_b, labels=labels, colors=colours_b, alpha=0.85)
ax.set_ylabel('Biomass (gC/m³)', fontsize=10)
ax.set_title('(a) Baseline — ecotype carbon biomass', fontsize=10, loc='left')
ax.legend(fontsize=9, loc='upper left')
ax.tick_params(labelsize=9)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())

# Panel (b) FPV_02 — stacked
ax = axes[1]
stack_f = []
for label in ECOTYPE_SUFFIXES.keys():
    series = ecotype_biomass['FPV_02'][label]
    if series.empty:
        continue
    stack_f.append(series.values)
times_f = ecotype_biomass['FPV_02'][list(ECOTYPE_SUFFIXES.keys())[0]].index
ax.stackplot(times_f, *stack_f, labels=labels, colors=colours_b, alpha=0.85)
ax.set_title('(b) FPV — ecotype carbon biomass', fontsize=10, loc='left')
ax.tick_params(labelsize=9)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())

# Panel (c) Δ (FPV − Baseline) — lines, per ecotype
ax = axes[2]
common_idx = ecotype_biomass['Baseline'][labels[0]].index.intersection(
    ecotype_biomass['FPV_02'][labels[0]].index)
for label in labels:
    base = ecotype_biomass['Baseline'][label].loc[common_idx]
    fpv  = ecotype_biomass['FPV_02'][label].loc[common_idx]
    diff = fpv - base
    ax.plot(diff.index, diff.values, color=ECOTYPE_COLOURS[label],
            lw=1.3, label=label)
ax.axhline(0, color='k', lw=0.7, ls='--', alpha=0.5)
ax.set_ylabel('Δ FPV − Baseline (gC/m³)', fontsize=10)
ax.set_title('(c) Δ ecotype biomass (FPV − Baseline)',
             fontsize=10, loc='left')
ax.tick_params(labelsize=9)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())

# Common x-limit
for ax in axes:
    ax.set_xlim(pd.Timestamp('2023-04-01'), pd.Timestamp('2023-08-31'))

plt.tight_layout()

# AFTER tight_layout settles, override panel (b)'s position
fig.subplots_adjust(left=0.05, right=0.98, top=0.92, bottom=0.12,
                    wspace=0.15)
pos_a = axes[0].get_position()
pos_b = axes[1].get_position()
axes[1].set_position([pos_a.x0 + pos_a.width + 0.01,
                      pos_b.y0, pos_b.width, pos_b.height])

save_fig(fig, '09_ecotype_biomass_baseline_vs_fpv02')
plt.show()

# Print summary stats
print('\nEcotype biomass summary — perimeter mean, Apr-Aug 2023:')
print(f'  {"Ecotype":<22} {"Baseline mean":>14} {"FPV_02 mean":>14} '
      f'{"Δ abs":>10} {"Δ %":>10}')
print('  ' + '-' * 80)
for label in labels:
    bm = ecotype_biomass['Baseline'][label].mean()
    fm = ecotype_biomass['FPV_02'][label].mean()
    d_abs = fm - bm
    d_pct = (d_abs / bm * 100) if bm != 0 else np.nan
    print(f'  {label:<22} {bm:>14.4f} {fm:>14.4f} '
          f'{d_abs:>+10.4f} {d_pct:>+9.2f}%')

In [ ]:
# Check 1: Is total carbon biomass higher under FPV, or just Chla?
b_biomass_jul = series_daily_surf['Baseline']['TotAlgae'].loc['2023-07'].mean()
f_biomass_jul = series_daily_surf['FPV_02']['TotAlgae'].loc['2023-07'].mean()
b_chla_jul    = series_daily_surf['Baseline']['Chlfa'].loc['2023-07'].mean()
f_chla_jul    = series_daily_surf['FPV_02']['Chlfa'].loc['2023-07'].mean()
print(f'July surface biomass (gC/m³): Baseline={b_biomass_jul:.3f}, '
      f'FPV={f_biomass_jul:.3f}, ratio={f_biomass_jul/b_biomass_jul:.2f}')
print(f'July surface Chla (µg/L):     Baseline={b_chla_jul:.3f}, '
      f'FPV={f_chla_jul:.3f}, ratio={f_chla_jul/b_chla_jul:.2f}')

In [ ]:
# Check 2: Species composition shift — what fraction of biomass is P-adapted?
date_range = slice('2023-07-01', '2023-07-31')
for scen in ['Baseline', 'FPV_02']:
    df = series_daily_surf[scen].loc[date_range]
    e_total = df[[c for c in df.columns if c.endswith('_E')]].sum(axis=1).mean()
    n_total = df[[c for c in df.columns if c.endswith('_N')]].sum(axis=1).mean()
    p_total = df[[c for c in df.columns if c.endswith('_P')]].sum(axis=1).mean()
    total = e_total + n_total + p_total
    print(f'{scen}:')
    print(f'  E (light-adapted): {e_total:.3f} ({e_total/total*100:.0f}%)')
    print(f'  N (N-adapted):     {n_total:.3f} ({n_total/total*100:.0f}%)')
    print(f'  P (P-adapted):     {p_total:.3f} ({p_total/total*100:.0f}%)')

In [ ]:
# Check 3: Depth-averaged biomass — is the column-integrated effect smaller?
b_biomass_davg = series_daily['Baseline']['TotAlgae'].loc['2023-07'].mean()
f_biomass_davg = series_daily['FPV_02']['TotAlgae'].loc['2023-07'].mean()
print(f'Layer-mean July biomass (gC/m³): Baseline={b_biomass_davg:.3f}, '
      f'FPV={f_biomass_davg:.3f}, '
      f'ratio={f_biomass_davg/b_biomass_davg:.2f}')

## Section 10 — Depth profiles (perimeter-averaged Hovmöller)

For each substance: a four-panel figure of time × depth Hovmöllers per
scenario, plus three FPV − Baseline difference panels. Computed directly
via xarray: `ds[sub].where(perim_da).mean(['M', 'N'])` over the K=K_TOP..K_BOT
range.

In [ ]:
# Use the canonical substance list from series_daily (which has the full set)
DERIVED = {'TotN', 'TotP', 'TotAlgae', 'Chlfa', 'AlgN', 'AlgP'}
direct_subs = [s for s in series_daily['Baseline'].columns if s not in DERIVED]
print(f'Building depth profiles for {len(direct_subs)} substances')

def compute_depth_profile(ds, substance, perim_da):
    """Build (T, K_ACTIVE) time × layer matrix, perimeter-averaged per layer."""
    if substance not in ds:
        return None
    da = ds[substance].sel(K=slice(K_TOP, K_BOT)).where(perim_da)
    if substance == 'OXY':
        da = da.where(da <= OXY_MAX_PHYS)
    # Mean over M and N at each (time, K) — collapses spatial dims
    da = da.mean(['M', 'N'])
    # Daily resample after spatial reduction (cheap once spatial is collapsed)
    da = da.resample(time='D').mean()
    return da

print('Computing depth profiles per substance per scenario...')
depth_profiles = {}
for scen, ds in runs.items():
    print(f'  {scen}...')
    depth_profiles[scen] = {}
    for sub in direct_subs:
        prof = compute_depth_profile(ds, sub, perim_da)
        if prof is not None:
            depth_profiles[scen][sub] = prof

# Reconstructed Chlfa per layer
print('\nReconstructing Chlfa depth profiles...')
chlfa_profiles = {}
for scen in SCEN_ORDER:
    chlfa_da = None
    for sp, ratio in CHLA_C.items():
        if sp not in depth_profiles[scen]:
            continue
        contrib = depth_profiles[scen][sp] * ratio * 1000.0
        chlfa_da = contrib if chlfa_da is None else (chlfa_da.fillna(0) + contrib.fillna(0))
    chlfa_profiles[scen] = chlfa_da
print('Done.')

In [ ]:
def plot_depth_panel_grid(profiles, substance, surface_mid_series=None,
                          cmap='viridis', diff_cmap='RdBu_r', save_name=None):
    """One figure: top row absolute (4 scenarios), bottom row 3 diffs.
    Panel letters (a)-(g) annotate each panel; scenario name added to top-left
    text. Substance shown only in suptitle and colorbars."""
    if substance not in profiles['Baseline'] or profiles['Baseline'][substance] is None:
        print(f'  skip {substance}: not in profiles')
        return

    prof_base = profiles['Baseline'][substance]
    times_b = pd.DatetimeIndex(prof_base.time.values)
    k_range = list(prof_base.K.values)

    prof_by_scen = {scen: profiles[scen][substance].values for scen in SCEN_ORDER}

    all_vals = np.concatenate([p[~np.isnan(p)] for p in prof_by_scen.values()])
    if all_vals.size == 0:
        print(f'  skip {substance}: no valid data'); return
    vmin, vmax = np.nanpercentile(all_vals, [2, 98])

    p50 = np.nanpercentile(all_vals, 50)
    rng = vmax - vmin
    skew = abs((p50 - (vmin + vmax) / 2) / rng) if rng > 0 else 0
    if skew > 0.35 and vmax > 0 and vmin >= 0:
        linthresh = max(vmax * 1e-3, 1e-9)
        norm = SymLogNorm(linthresh=linthresh, vmin=vmin, vmax=vmax, base=10)
    else:
        norm = Normalize(vmin=vmin, vmax=vmax)

    thicknesses = np.array([THICKNESS_M[k] for k in k_range])
    d_edges = np.zeros(len(k_range) + 1)
    for i in range(len(k_range)):
        d_edges[i + 1] = d_edges[i] + thicknesses[i]

    t_num = mdates.date2num(times_b.to_pydatetime())
    if len(t_num) > 1:
        dt = t_num[1] - t_num[0]
        t_edges = np.concatenate([t_num - dt/2, [t_num[-1] + dt/2]])
    else:
        t_edges = np.array([t_num[0] - 0.5, t_num[0] + 0.5])

    base_vals = prof_by_scen['Baseline']
    diffs = {scen: prof_by_scen[scen] - base_vals
             for scen in ['FPV_02', 'FPV_03', 'FPV_04']}
    diff_vals = np.concatenate([d[~np.isnan(d)] for d in diffs.values()])
    dlim = np.nanpercentile(np.abs(diff_vals), 98) if diff_vals.size else 1.0

    cm_main = plt.get_cmap(cmap).copy();      cm_main.set_bad('#dddddd')
    cm_diff = plt.get_cmap(diff_cmap).copy(); cm_diff.set_bad('#dddddd')

    display_label = label_for(substance)

    fig, axes = plt.subplots(2, 4, figsize=(9, 5), sharex=True, sharey=True)

    panel_letters = list('abcdefgh')
    # Top row: absolute (a, b, c, d)
    for ax, scen, letter in zip(axes[0], SCEN_ORDER, panel_letters[:4]):
        prof = prof_by_scen[scen]
        pc = ax.pcolormesh(t_edges, d_edges, np.ma.masked_invalid(prof.T),
                           cmap=cm_main, norm=norm, shading='flat')
        if surface_mid_series is not None:
            ax.plot(mdates.date2num(surface_mid_series.index.to_pydatetime()),
                    surface_mid_series.values, color='black', lw=0.7, alpha=0.85)
        ax.invert_yaxis()
        ax.tick_params(labelsize=8)
        add_panel_letter(ax, f'({letter}) {scen}', x=0.02, y=0.97, fontsize=9)
    axes[0, 0].set_ylabel('Depth (m)', fontsize=10)
    cbar_ax = fig.add_axes([0.92, 0.55, 0.012, 0.32])
    fig.colorbar(pc, cax=cbar_ax, label=display_label)

    # Bottom row: differences (e, f, g) — leftmost slot stays empty
    # Bottom row: differences (e, f, g) — leftmost slot stays empty
    axes[1, 0].set_visible(False)
    diff_letters = panel_letters[4:7]
    bottom_axes = [axes[1, 1], axes[1, 2], axes[1, 3]]
    for ax, scen, letter in zip(bottom_axes, ['FPV_02', 'FPV_03', 'FPV_04'],   # CHANGED: FPV_04 was missing
                                 diff_letters):
        d = diffs[scen]
        pcd = ax.pcolormesh(t_edges, d_edges, np.ma.masked_invalid(d.T),
                            cmap=cm_diff, vmin=-dlim, vmax=dlim, shading='flat')
        if surface_mid_series is not None:
            ax.plot(mdates.date2num(surface_mid_series.index.to_pydatetime()),
                    surface_mid_series.values, color='black', lw=0.7, alpha=0.85)
        ax.invert_yaxis()
        ax.tick_params(labelsize=8)
        add_panel_letter(ax, f'({letter}) Δ ({scen} − Baseline)',
                         x=0.02, y=0.1, fontsize=9)
    
    # Show y-tick labels ONLY on the leftmost bottom panel (panel e = axes[1, 1])
    bottom_axes[0].tick_params(labelleft=True)
    bottom_axes[0].set_ylabel('Depth (m)', fontsize=10)
    cbar_ax2 = fig.add_axes([0.92, 0.12, 0.012, 0.32])
    fig.colorbar(pcd, cax=cbar_ax2, label=f'Δ {display_label}')

    for ax in axes[1, :]:
        if ax.get_visible():
            ax.xaxis_date()
            ax.xaxis.set_major_locator(mdates.MonthLocator())
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))

    # Reduced hspace from 0.30 → 0.12 to tighten gap between rows
    plt.subplots_adjust(left=0.05, right=0.90, top=0.88, bottom=0.10,
                        hspace=0.12, wspace=0.10)
    if save_name:
        save_fig(fig, save_name)
    plt.show()


# Build per-species-group depth profiles (sum all ecotypes per group)
SPECIES_ECOTYPES_DEPTH = {
    'Diatoms':       ['FDIATOMS_E', 'FDIATOMS_N', 'FDIATOMS_P'],
    'Greens':        ['GREENS_E',   'GREENS_N',   'GREENS_P'],
    'Flagellates':   ['FFLAGELA'],
    'Cyanobacteria': ['MICROCYS_E', 'MICROCYS_N', 'MICROCYS_P'],
}

print('Building per-species-group depth profiles (summing across ecotypes)...')
group_profiles = {scen: {} for scen in SCEN_ORDER}
for group_name, ecotypes in SPECIES_ECOTYPES_DEPTH.items():
    for scen in SCEN_ORDER:
        present = [s for s in ecotypes if s in depth_profiles[scen]]
        if not present:
            print(f'  {scen} | {group_name}: no ecotypes available, skipping')
            continue
        # Sum the (time, K) profiles across ecotypes
        summed = sum(depth_profiles[scen][s].fillna(0) for s in present)
        group_profiles[scen][group_name] = summed
print('Done.')

# Plot the individual substances (non-algae state variables)
NON_ALGAE_SUBS = [s for s in SUBSTANCES
                  if s not in sum(SPECIES_ECOTYPES_DEPTH.values(), [])]
print(f'\nPlotting depth profiles for {len(NON_ALGAE_SUBS)} non-algae substances...')
for sub in NON_ALGAE_SUBS:
    plot_depth_panel_grid(depth_profiles, sub,
                          surface_mid_series=surface_mid_depth_series,
                          save_name=f'10_depth_{sub}')

# Plot the aggregated species groups
print(f'\nPlotting depth profiles for {len(SPECIES_ECOTYPES_DEPTH)} species groups...')
for group_name in SPECIES_ECOTYPES_DEPTH:
    plot_depth_panel_grid(group_profiles, group_name,
                          cmap='YlGn',
                          surface_mid_series=surface_mid_depth_series,
                          save_name=f'10_depth_{group_name}')

# Reconstructed Chlfa (unchanged)
chlfa_pseudo = {scen: {'Chlfa': chlfa_profiles[scen]} for scen in SCEN_ORDER}
plot_depth_panel_grid(chlfa_pseudo, 'Chlfa',
                      cmap='YlGn',
                      surface_mid_series=surface_mid_depth_series,
                      save_name='10_depth_Chlfa')

In [ ]:
# ── Single-substance Baseline vs FPV_02 depth profile (3-panel) ──────────
# Mirrors the DO depth-profile layout from the diagnostic notebook:
# (a) Baseline, (b) FPV_02, (c) Δ — all sharing y-axis, two shared colorbars.

def plot_depth_panel_3col(profiles, substance, scenario='FPV_02',
                           cmap='viridis', diff_cmap='RdBu_r',
                           save_name=None,
                           surface_mid_series=None, ref_name='Baseline'):
    """Single-substance 3-panel depth-time figure: Baseline | Scenario | Δ."""
    if (substance not in profiles['Baseline']
            or profiles['Baseline'][substance] is None
            or substance not in profiles[scenario]
            or profiles[scenario][substance] is None):
        print(f'  skip {substance}: not in both Baseline and {scenario}'); return

    prof_B = profiles['Baseline'][substance]
    prof_S = profiles[scenario][substance]
    times_b = pd.DatetimeIndex(prof_B.time.values)
    k_range = list(prof_B.K.values)

    base_vals = prof_B.values
    scen_vals = prof_S.values
    diff_vals = scen_vals - base_vals

    # Pooled colour range across Baseline + scenario (absolute panels)
    all_abs = np.concatenate([
        base_vals[~np.isnan(base_vals)],
        scen_vals[~np.isnan(scen_vals)],
    ])
    if all_abs.size == 0:
        print(f'  skip {substance}: no valid data'); return
    vmin, vmax = np.nanpercentile(all_abs, [2, 98])

    # Auto-detect skew → SymLog (same heuristic as the 4-scenario grid)
    p50 = np.nanpercentile(all_abs, 50)
    rng = vmax - vmin
    skew = abs((p50 - (vmin + vmax) / 2) / rng) if rng > 0 else 0
    if skew > 0.35 and vmax > 0 and vmin >= 0:
        linthresh = max(vmax * 1e-3, 1e-9)
        norm = SymLogNorm(linthresh=linthresh, vmin=vmin, vmax=vmax, base=10)
    else:
        norm = Normalize(vmin=vmin, vmax=vmax)

    # Difference range — symmetric 98%
    dvalid = diff_vals[~np.isnan(diff_vals)]
    dlim = np.nanpercentile(np.abs(dvalid), 98) if dvalid.size else 1.0

    # Depth edges (consistent with the 4-scenario grid)
    thicknesses = np.array([THICKNESS_M[k] for k in k_range])
    d_edges = np.zeros(len(k_range) + 1)
    for i in range(len(k_range)):
        d_edges[i + 1] = d_edges[i] + thicknesses[i]

    # Time edges
    t_num = mdates.date2num(times_b.to_pydatetime())
    if len(t_num) > 1:
        dt = t_num[1] - t_num[0]
        t_edges = np.concatenate([t_num - dt/2, [t_num[-1] + dt/2]])
    else:
        t_edges = np.array([t_num[0] - 0.5, t_num[0] + 0.5])

    cm_main = plt.get_cmap(cmap).copy();      cm_main.set_bad('#dddddd')
    cm_diff = plt.get_cmap(diff_cmap).copy(); cm_diff.set_bad('#dddddd')

    display_label = label_for(substance)

    # ── Build figure: 3 panels in a row, shared y ───────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(9, 3), sharey=True)

    # Panel (a) Baseline
    pc_abs = axes[0].pcolormesh(t_edges, d_edges,
                                  np.ma.masked_invalid(base_vals.T),
                                  cmap=cm_main, norm=norm, shading='flat')
    axes[0].invert_yaxis()
    axes[0].set_ylabel('Depth (m)', fontsize=9)
    axes[0].tick_params(labelsize=8)

    # Panel (b) Scenario
    axes[1].pcolormesh(t_edges, d_edges,
                        np.ma.masked_invalid(scen_vals.T),
                        cmap=cm_main, norm=norm, shading='flat')
    axes[1].invert_yaxis()
    axes[1].tick_params(labelsize=8)

    # Panel (c) Δ
    pc_diff = axes[2].pcolormesh(t_edges, d_edges,
                                   np.ma.masked_invalid(diff_vals.T),
                                   cmap=cm_diff, vmin=-dlim, vmax=dlim,
                                   shading='flat')
    axes[2].invert_yaxis()
    axes[2].tick_params(labelsize=8)

    # Optional black surface-mid line overlay
    if surface_mid_series is not None:
        smt = mdates.date2num(surface_mid_series.index.to_pydatetime())
        for ax in axes:
            ax.plot(smt, surface_mid_series.values,
                    color='black', lw=0.7, alpha=0.85)

    # X-axis date formatting on all panels
    for ax in axes:
        ax.xaxis_date()
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))

    # Panel letters
    axes[0].text(0.02, 0.08, f'(a) {ref_name}', transform=axes[0].transAxes,
                  fontsize=9, ha='left', va='top',
                  bbox=dict(facecolor='white', edgecolor='none',
                            pad=2.0, alpha=0.85))
    axes[1].text(0.02, 0.08, f'(b) {scenario}', transform=axes[1].transAxes,
                  fontsize=9, ha='left', va='top',
                  bbox=dict(facecolor='white', edgecolor='none',
                            pad=2.0, alpha=0.85))
    axes[2].text(0.02, 0.08, f'(c) Δ ({scenario} − {ref_name})',
                  transform=axes[2].transAxes, fontsize=9, ha='left', va='top',
                  bbox=dict(facecolor='white', edgecolor='none',
                            pad=2.0, alpha=0.85))

    # Two shared colorbars — same layout as the DO diagnostic
    fig.subplots_adjust(left=0.06, right=0.86, top=0.95, bottom=0.18,
                        wspace=0.05)
    cbar_ax_abs  = fig.add_axes([0.87, 0.18, 0.012, 0.77])
    fig.colorbar(pc_abs, cax=cbar_ax_abs, label=display_label)
    cbar_ax_diff = fig.add_axes([0.96, 0.18, 0.012, 0.77])
    fig.colorbar(pc_diff, cax=cbar_ax_diff,
                 label=f'Δ {display_label}\n({scenario} − {ref_name})')

    if save_name:
        save_fig(fig, save_name)
    plt.show()


# Reconstructed Chlfa — Baseline vs FPV_02 only
plot_depth_panel_3col(chlfa_pseudo, 'Chlfa', scenario='FPV_02',
                       cmap='YlGn',
                       surface_mid_series=surface_mid_depth_series,
                       save_name='10_depth_Chlfa_baseline_vs_fpv02')

In [ ]:
# Get radiation values at ALL FPV-affected cells, pooled across time
b_at_fpv = rad['noFPV'][:, covered_surf_segs]   # (time, n_fpv_cells)
f_at_fpv = rad['02'][:, covered_surf_segs]

# Mean attenuation across all cells, all timesteps
overall_atten = (1 - f_at_fpv.mean() / b_at_fpv.mean()) * 100
print(f'Mean radiation attenuation across all 622 cells: {overall_atten:.1f}%')

# Distribution of per-cell attenuation (averaged over time)
per_cell_atten = (1 - f_at_fpv.mean(axis=0) / b_at_fpv.mean(axis=0)) * 100
print(f'Per-cell attenuation percentiles:')
print(f'  5th:   {np.percentile(per_cell_atten, 5):.1f}%')
print(f'  25th:  {np.percentile(per_cell_atten, 25):.1f}%')
print(f'  Median: {np.percentile(per_cell_atten, 50):.1f}%')
print(f'  75th:  {np.percentile(per_cell_atten, 75):.1f}%')
print(f'  95th:  {np.percentile(per_cell_atten, 95):.1f}%')
print(f'  Max:   {per_cell_atten.max():.1f}%')

In [ ]:
# Where in the water column is the FPV-increased biomass actually accumulating?
print('Biomass at each depth (gC/m³), July mean:')
for k in range(K_TOP, K_BOT + 1):
    b_k = (runs['Baseline']['FDIATOMS_E'] + runs['Baseline']['GREENS_E'] + 
           runs['Baseline']['MICROCYS_E'] + runs['Baseline']['FDIATOMS_P'] +
           runs['Baseline']['GREENS_P'] + runs['Baseline']['MICROCYS_P'])
    f_k = (runs['FPV_02']['FDIATOMS_E'] + runs['FPV_02']['GREENS_E'] +
           runs['FPV_02']['MICROCYS_E'] + runs['FPV_02']['FDIATOMS_P'] +
           runs['FPV_02']['GREENS_P'] + runs['FPV_02']['MICROCYS_P'])
    b_val = b_k.where(perim_da).sel(K=k).mean(['M','N']).resample(time='D').mean().loc['2023-07'].mean().values
    f_val = f_k.where(perim_da).sel(K=k).mean(['M','N']).resample(time='D').mean().loc['2023-07'].mean().values
    print(f'  K={k}: Baseline={float(b_val):.3f}, FPV={float(f_val):.3f}, '
          f'ratio={float(f_val)/float(b_val):.2f}')

## Section 11 — Surface maps (snapshot dates)

For each substance × each snapshot date: four-panel figure of K=K_TOP (active
surface) across baseline + 3 FPV runs, plus difference maps below.

In [ ]:
import matplotlib.gridspec as gridspec

# Species groups (sum across all ecotypes per group)
SPECIES_ECOTYPES_MAP = {
    'Diatoms':       ['FDIATOMS_E', 'FDIATOMS_N', 'FDIATOMS_P'],
    'Greens':        ['GREENS_E',   'GREENS_N',   'GREENS_P'],
    'Flagellates':   ['FFLAGELA'],
    'Cyanobacteria': ['MICROCYS_E', 'MICROCYS_N', 'MICROCYS_P'],
}


def get_surface_field_at_date(ds, substance, target_date):
    """Get surface (K=K_TOP) field as a daily mean on the target date.
    Handles substance keys that are species groups (sum ecotypes) or direct
    substance names."""
    if substance in SPECIES_ECOTYPES_MAP:
        # Aggregate across all ecotypes in the group
        ecotypes = SPECIES_ECOTYPES_MAP[substance]
        present = [s for s in ecotypes if s in ds.data_vars]
        if not present:
            return None
        da = sum(ds[s].sel(K=K_TOP) for s in present)
    else:
        if substance not in ds.data_vars:
            return None
        da = ds[substance].sel(K=K_TOP)

    # Daily-average then pick the target date
    da = da.resample(time='D').mean()
    da = da.sel(time=target_date, method='nearest')

    if substance == 'OXY':
        da = da.where(da <= OXY_MAX_PHYS)
    return da


def plot_surface_maps_at_date(runs, substance, date_str,
                              cmap='viridis', diff_cmap='RdBu_r',
                              save_name=None):
    """Plot surface maps for one substance × one snapshot date.
    Top row: absolute values per scenario (a-d).
    Bottom row: FPV − Baseline difference panels (e-g).
    Daily-averaged; species substances are summed across ecotypes."""
    target = pd.Timestamp(date_str)
    panels = []
    for scen in SCEN_ORDER:
        da = get_surface_field_at_date(runs[scen], substance, target)
        if da is None:
            print(f'{substance} not in {scen}'); return
        panels.append((scen, da.values, pd.Timestamp(da.time.values)))

    base_name, base_surf, t_actual = panels[0]
    n_runs = len(panels)
    display_label = label_for(substance)

    all_v = np.concatenate([p[1][~np.isnan(p[1])] for p in panels])
    if all_v.size == 0:
        print(f'No valid surface data for {substance} on {date_str}')
        return
    vmin, vmax = np.nanpercentile(all_v, [2, 98])
    cm = plt.get_cmap(cmap).copy(); cm.set_bad('#dddddd')
    cm_d = plt.get_cmap(diff_cmap).copy(); cm_d.set_bad('#dddddd')

    diffs = [p[1] - base_surf for p in panels[1:]]
    diff_vals = np.concatenate([d[~np.isnan(d)] for d in diffs])
    has_diffs = diff_vals.size > 0
    dlim = np.nanpercentile(np.abs(diff_vals), 98) if has_diffs else 1.0

    fig = plt.figure(figsize=(3.5 * n_runs + 1.5, 6.5))
    gs = gridspec.GridSpec(
        2, n_runs + 1, figure=fig,
        width_ratios=[1] * n_runs + [0.06],
        height_ratios=[1, 1],
        wspace=0.20, hspace=0.20,
    )

    panel_letters = list('abcdefgh')

    im_abs = None
    for j, ((name, surf, ts), letter) in enumerate(zip(panels, panel_letters[:4])):
        ax = fig.add_subplot(gs[0, j])
        im_abs = ax.pcolormesh(np.ma.masked_invalid(surf.T),
                               cmap=cm, vmin=vmin, vmax=vmax)
        ax.set_aspect('equal'); ax.tick_params(labelsize=7)
        add_panel_letter(ax, f'({letter}) {name}  {ts.date()}',
                         x=0.02, y=0.97, fontsize=9)

    im_diff = None
    diff_letters = panel_letters[4:7]
    for j, ((name, _, _), letter) in enumerate(zip(panels[1:], diff_letters),
                                                 start=1):
        ax = fig.add_subplot(gs[1, j])
        diff = panels[j][1] - base_surf
        im_diff = ax.pcolormesh(np.ma.masked_invalid(diff.T),
                                cmap=cm_d, vmin=-dlim, vmax=dlim)
        ax.set_aspect('equal'); ax.tick_params(labelsize=7)
        add_panel_letter(ax,
                         f'({letter}) Δ({name} − {base_name})',
                         x=0.02, y=0.97, fontsize=8)

    cax_abs = fig.add_subplot(gs[0, -1])
    fig.colorbar(im_abs, cax=cax_abs, label=display_label)
    if im_diff is not None:
        cax_diff = fig.add_subplot(gs[1, -1])
        fig.colorbar(im_diff, cax=cax_diff,
                     label=f'Δ(FPV scen − Baseline)')

    # fig.suptitle(f'{display_label} — surface (K={K_TOP}) — {t_actual.date()}',
    #              fontsize=12, y=0.98)

    if save_name:
        save_fig(fig, save_name)
    plt.show()


# Build list of substances to plot: non-algae (individual) + species groups
NON_ALGAE_SUBS = [s for s in SUBSTANCES
                  if s not in sum(SPECIES_ECOTYPES_MAP.values(), [])]
GROUP_SUBS = list(SPECIES_ECOTYPES_MAP.keys())
ALL_SUBS = NON_ALGAE_SUBS + GROUP_SUBS

print(f'Plotting surface maps for {len(NON_ALGAE_SUBS)} non-algae substances '
      f'+ {len(GROUP_SUBS)} species groups...')

for sub in ALL_SUBS:
    # Skip if substance / group has no representation in Baseline
    if sub in SPECIES_ECOTYPES_MAP:
        if not any(s in runs['Baseline'].data_vars
                   for s in SPECIES_ECOTYPES_MAP[sub]):
            continue
    else:
        if sub not in runs['Baseline'].data_vars:
            continue
    for date_str in SNAPSHOT_DATES:
        plot_surface_maps_at_date(runs, sub, date_str,
                                  save_name=f'11_map_{sub}_{date_str.replace("-", "")}')

### 11.2 — Maps of quantities that are not in the `.map` (CHANGED, new)
Total algae, TotN and TotP are computed per cell from the `.map` substances with the same definitions as
Section 6 (Chl a follows in 11.3). Each quantity uses one colour range for all snapshot dates and scenarios, so
the maps can be compared with each other.

In [ ]:
def derived_surface_at_date(ds, key, target_date):
    """Surface (K=K_TOP) daily-mean field of a derived quantity on target_date.
    key: 'TotAlgae', 'Chlfa', 'AlgN', 'AlgP', 'TotN' or 'TotP' (definitions as in add_derived)."""
    def surf(sp):
        da = ds[sp].sel(K=K_TOP).resample(time='D').mean()
        return da.sel(time=pd.Timestamp(target_date), method='nearest')
    alg = [s for s in ALGAE if s in ds.data_vars]
    weights = {'TotAlgae': {s: 1.0 for s in alg}, 'Chlfa': {s: CHLA_C[s] * 1000.0 for s in alg},
               'AlgN': {s: NCR_ALG[s] for s in alg}, 'AlgP': {s: PCR_ALG[s] for s in alg}}
    if key in weights:
        return sum(surf(s) * w for s, w in weights[key].items())
    if key == 'TotN':
        return surf('NH4') + surf('NO3') + surf('PON1') + derived_surface_at_date(ds, 'AlgN', target_date)
    if key == 'TotP':
        return surf('PO4') + surf('POP1') + derived_surface_at_date(ds, 'AlgP', target_date)
    raise KeyError(key)


def plot_derived_maps_at_date(key, date_str, vmin, vmax, dlim, cmap='viridis', save_name=None):
    """Same layout as the substance maps: top row the 4 scenarios, bottom row FPV − Baseline."""
    panels = [(scen, derived_surface_at_date(runs[scen], key, date_str)) for scen in SCEN_ORDER]
    base = panels[0][1].values
    n_runs = len(panels)
    cm = plt.get_cmap(cmap).copy(); cm.set_bad('#dddddd')
    cm_d = plt.get_cmap('RdBu_r').copy(); cm_d.set_bad('#dddddd')
    fig = plt.figure(figsize=(3 * n_runs + 1.5, 7))
    gs = gridspec.GridSpec(2, n_runs + 1, figure=fig, width_ratios=[1] * n_runs + [0.06],
                           wspace=0.20, hspace=0.10)
    letters = list('abcdefgh')
    ax0 = None
    for j, (name, da) in enumerate(panels):
        ax = fig.add_subplot(gs[0, j], sharex=ax0, sharey=ax0)
        ax0 = ax0 or ax
        im_abs = ax.pcolormesh(np.ma.masked_invalid(da.values.T), cmap=cm, vmin=vmin, vmax=vmax)
        ax.set_aspect('equal'); ax.tick_params(labelsize=7)
        if j:
            plt.setp(ax.get_yticklabels(), visible=False)
        add_panel_letter(ax, f'({letters[j]}) {name}  {pd.Timestamp(da.time.values).date()}', fontsize=9)
    for j, (name, da) in enumerate(panels[1:], start=1):
        ax = fig.add_subplot(gs[1, j], sharex=ax0, sharey=ax0)
        im_diff = ax.pcolormesh(np.ma.masked_invalid((da.values - base).T), cmap=cm_d, vmin=-dlim, vmax=dlim)
        ax.set_aspect('equal'); ax.tick_params(labelsize=7)
        if j > 1:
            plt.setp(ax.get_yticklabels(), visible=False)
        add_panel_letter(ax, f'({letters[n_runs + j - 1]}) Δ({name} − Baseline)', fontsize=8)
    fig.colorbar(im_abs, cax=fig.add_subplot(gs[0, -1]), label=label_for(key))
    fig.colorbar(im_diff, cax=fig.add_subplot(gs[1, -1]), label='Δ(FPV scenario − Baseline)')
    if save_name:
        save_fig(fig, save_name)
    plt.show()


DERIVED_MAP_KEYS = {'TotAlgae': 'YlGn', 'TotN': 'YlGnBu', 'TotP': 'viridis'}
for key, cmap in DERIVED_MAP_KEYS.items():
    vals, dvals = [], []
    fields = {d: {scen: derived_surface_at_date(runs[scen], key, d).values for scen in SCEN_ORDER}
              for d in SNAPSHOT_DATES}
    for d, f in fields.items():
        for scen in SCEN_ORDER:
            vals.append(f[scen][np.isfinite(f[scen])])
            if scen != 'Baseline':
                dd = f[scen] - f['Baseline']; dvals.append(dd[np.isfinite(dd)])
    vmin, vmax = np.nanpercentile(np.concatenate(vals), [2, 98])
    dlim = np.nanpercentile(np.abs(np.concatenate(dvals)), 98) or 1.0
    print(f'{key}: colour range {vmin:.3g} – {vmax:.3g}, differences ± {dlim:.3g}')
    for d in SNAPSHOT_DATES:
        plot_derived_maps_at_date(key, d, vmin, vmax, dlim, cmap=cmap,
                                  save_name=f'11_map_{key}_{d.replace("-", "")}')

### 11.3 — Chl a maps (Σ biomass × CHLACALG, checked in 6.0)

In [ ]:
# ── Chla surface maps (reconstructed from algae ecotypes × ChlaCAlg) ────
# Chla is not a .map substance — we reconstruct it on the fly per snapshot
# date. Daily-averaged before sampling the target date so the snapshot
# represents a 24-hour mean rather than an arbitrary 6-hour timestep.
def _chla_surface_at_date(ds, date_str):
    """Reconstruct surface (K=K_TOP) Chla at a target date, in µg/L.
    Each ecotype's contribution is daily-averaged before sampling."""
    target = pd.Timestamp(date_str)
    chla_surf = None
    t_actual = None
    for sp, ratio in CHLA_C.items():
        if sp not in ds:
            continue
        # Daily-average at K=K_TOP, then sample the target date
        da = ds[sp].sel(K=K_TOP).resample(time='D').mean()
        da = da.sel(time=target, method='nearest')
        contrib = da.values * ratio * 1000.0
        chla_surf = contrib if chla_surf is None else chla_surf + np.nan_to_num(contrib)
        t_actual = pd.Timestamp(da.time.values)
    return chla_surf, t_actual


# ── Pre-compute global value range across all dates × all scenarios ─────
# So every Chla map shares the same colorbar (and the differences too).
print('Pre-computing global Chla range across all snapshot dates...')
all_chla_vals = []
all_chla_diffs = []
for date_str in SNAPSHOT_DATES:
    base_surf = None
    for scen in SCEN_ORDER:
        surf, _ = _chla_surface_at_date(runs[scen], date_str)
        if surf is None:
            continue
        valid = surf[~np.isnan(surf)]
        all_chla_vals.append(valid)
        if scen == 'Baseline':
            base_surf = surf
        elif base_surf is not None:
            diff_valid = (surf - base_surf)
            all_chla_diffs.append(diff_valid[~np.isnan(diff_valid)])

if all_chla_vals:
    vals_concat = np.concatenate(all_chla_vals)
    CHLA_VMIN, CHLA_VMAX = np.nanpercentile(vals_concat, [2, 98])
    print(f'  Chla absolute range:  {CHLA_VMIN:.2f} - {CHLA_VMAX:.2f} µg/L')
else:
    CHLA_VMIN, CHLA_VMAX = 0, 50
    print(f'  No data — using fallback')

if all_chla_diffs:
    diffs_concat = np.concatenate(all_chla_diffs)
    CHLA_DLIM = np.nanpercentile(np.abs(diffs_concat), 98)
    print(f'  Chla difference range: ±{CHLA_DLIM:.2f} µg/L')
else:
    CHLA_DLIM = 1.0


def plot_chla_surface_maps_at_date(runs, date_str,
                                    vmin=None, vmax=None, dlim=None,
                                    cmap='YlGn', diff_cmap='RdBu_r',
                                    save_name=None):
    """Surface (K=K_TOP) Chla map across 4 scenarios + 3 differences.
    If vmin/vmax/dlim are None, range is computed per-figure (legacy).
    Pass values to lock the range across multiple figures."""
    panels = []
    t_actual = None
    for scen in SCEN_ORDER:
        surf, t = _chla_surface_at_date(runs[scen], date_str)
        if surf is None:
            print(f'No Chla ecotypes in {scen}'); return
        panels.append((scen, surf, t))
        t_actual = t

    base_name, base_surf, t_actual = panels[0]
    n_runs = len(panels)
    display_label = label_for('Chlfa')

    if vmin is None or vmax is None:
        all_v = np.concatenate([p[1][~np.isnan(p[1])] for p in panels])
        if all_v.size == 0:
            print(f'No valid Chla data on {date_str}'); return
        vmin, vmax = np.nanpercentile(all_v, [2, 98])

    cm = plt.get_cmap(cmap).copy(); cm.set_bad('#dddddd')
    cm_d = plt.get_cmap(diff_cmap).copy(); cm_d.set_bad('#dddddd')

    diffs = [p[1] - base_surf for p in panels[1:]]
    diff_vals = np.concatenate([d[~np.isnan(d)] for d in diffs])
    has_diffs = diff_vals.size > 0
    if dlim is None:
        dlim = np.nanpercentile(np.abs(diff_vals), 98) if has_diffs else 1.0

    fig = plt.figure(figsize=(3 * n_runs + 1.5, 7))
    gs = gridspec.GridSpec(
        2, n_runs + 1, figure=fig,
        width_ratios=[1] * n_runs + [0.06],
        height_ratios=[1, 1],
        wspace=0.20, hspace=0.10,
    )

    panel_letters = list('abcdefgh')

    im_abs = None
    ax0 = None
    for j, ((name, surf, ts), letter) in enumerate(zip(panels, panel_letters[:4])):
        sharex = ax0 if j > 0 else None
        sharey = ax0 if j > 0 else None
        ax = fig.add_subplot(gs[0, j], sharex=sharex, sharey=sharey)
        if j == 0:
            ax0 = ax
        im_abs = ax.pcolormesh(np.ma.masked_invalid(surf.T),
                               cmap=cm, vmin=vmin, vmax=vmax)
        ax.set_aspect('equal'); ax.tick_params(labelsize=7)
        if j > 0:
            plt.setp(ax.get_yticklabels(), visible=False)
        add_panel_letter(ax, f'({letter}) {name}  {ts.date()}',
                         x=0.02, y=0.97, fontsize=9)

    im_diff = None
    diff_letters = panel_letters[4:7]
    ax_diff0 = None
    for j, ((name, _, _), letter) in enumerate(zip(panels[1:], diff_letters), start=1):
        sharex_ax = fig.axes[j]
        sharey_ax = ax_diff0
        ax = fig.add_subplot(gs[1, j],
                             sharex=sharex_ax,
                             sharey=sharey_ax)
        if ax_diff0 is None:
            ax_diff0 = ax
        diff = panels[j][1] - base_surf
        im_diff = ax.pcolormesh(np.ma.masked_invalid(diff.T),
                                cmap=cm_d, vmin=-dlim, vmax=dlim)
        ax.set_aspect('equal'); ax.tick_params(labelsize=7)
        if j > 1:
            plt.setp(ax.get_yticklabels(), visible=False)
        add_panel_letter(ax,
                         f'({letter}) Δ({name} − {base_name})',
                         x=0.02, y=0.97, fontsize=8)

    cax_abs = fig.add_subplot(gs[0, -1])
    fig.colorbar(im_abs, cax=cax_abs, label=display_label)
    if im_diff is not None:
        cax_diff = fig.add_subplot(gs[1, -1])
        fig.colorbar(im_diff, cax=cax_diff,
                     label=f'Δ(FPV scenario − Baseline)')

    if save_name:                       # CHANGED: the figure was never saved or shown
        save_fig(fig, save_name)
    plt.show()


# Loop over snapshot dates — same colorbar range on every figure
for date_str in SNAPSHOT_DATES:
    plot_chla_surface_maps_at_date(runs, date_str,
                                    vmin=CHLA_VMIN, vmax=CHLA_VMAX,
                                    dlim=CHLA_DLIM,
                                    save_name=f'11_map_Chlfa_{date_str.replace("-", "")}')

## Section 12 — Shading-only test (run 08): what the panels' light and wind reduction do on their own

Run 08 uses the **baseline hydrodynamics** (water level, temperature, mixing of the no-FPV FLOW run) with the
**FPV_02 light and wind** in WAQ. Comparing three runs separates the two pathways:

| Comparison | Isolates |
|---|---|
| Shading only − Baseline | less light and wind in WAQ, identical hydrodynamics |
| FPV_02 − Shading only | the FPV_02 FLOW hydrodynamics (water level, temperature, mixing), identical WAQ forcing |
| FPV_02 − Baseline | the full FPV_02 effect (sum of the two rows above) |

All quantities use the same definitions and the same FPV_02 perimeter as Sections 6–11.

In [ ]:
# ── Load run 08 and build its series with the same functions as Section 6 ──
runs_shade = {SHADE: open_scenario(SHADE_NC)[0]}
his_shade = read_his_file(SHADE_HIS)
print(f'{SHADE}: NetCDF {len(runs_shade[SHADE].time)} times; .his {len(his_shade["timestamps"])} records')

TRIO = ['Baseline', SHADE, 'FPV_02']
TRIO_LABEL = {'Baseline': 'Baseline', SHADE: 'Shading only', 'FPV_02': 'FPV_02'}
TRIO_DIFFS = [(SHADE, 'Baseline', 'Δ shading only'), ('FPV_02', SHADE, 'Δ hydrodynamics'),
              ('FPV_02', 'Baseline', 'Δ total')]

layer_trio = {s: series_daily[s] for s in ('Baseline', 'FPV_02')}
layer_trio[SHADE] = add_derived(perimeter_mean_series(runs_shade[SHADE], perim_da))
surf_trio = {s: series_daily_surf[s] for s in ('Baseline', 'FPV_02')}
surf_trio[SHADE] = add_derived(build_series_surface(runs_shade[SHADE], perim_da, direct_subs)).resample('D').mean()

# ── Table: Apr-Aug and Jun-Aug means, three runs and three differences ──
rows = []
for layer_name, d in (('Surface (K=7)', surf_trio), (LAYERMEAN_LABEL, layer_trio)):
    for sub in ['Chlfa', 'TotAlgae', 'OXY', 'NO3', 'NH4', 'PO4', 'TotN', 'TotP']:
        if not all(sub in d[s].columns for s in TRIO):
            continue
        for per, (t0, t1) in (('Apr-Aug', ('2023-04-01', '2023-08-31')), ('Jun-Aug', ('2023-06-01', '2023-08-31'))):
            v = {s: d[s][sub].loc[t0:t1].mean() for s in TRIO}
            row = {'Layer': layer_name, 'Variable': sub, 'Period': per, **{TRIO_LABEL[s]: v[s] for s in TRIO}}
            for a, b, lab in TRIO_DIFFS:
                row[lab] = v[a] - v[b]
                row[f'{lab} (%)'] = (v[a] - v[b]) / v[b] * 100 if v[b] else np.nan
            rows.append(row)
shade_tbl = pd.DataFrame(rows).set_index(['Layer', 'Variable', 'Period'])
shade_tbl.to_csv(os.path.join(RESULTS_DIR, '12_shading_only_summary.csv'))
with pd.option_context('display.width', 250, 'display.max_columns', 20):
    print(shade_tbl.round(4).to_string())

In [ ]:
# ── Time series: surface and layer mean, three runs ────────────────────
VARS_SHADE = ['Chlfa', 'TotAlgae', 'OXY', 'PO4']
fig, axes = plt.subplots(2, len(VARS_SHADE), figsize=(3.2 * len(VARS_SHADE), 5.5), sharex=True)
styles = {'Baseline': dict(color='k', ls='--'), SHADE: dict(color=SCEN_COLORS[SHADE], ls='-'),
          'FPV_02': dict(color=SCEN_COLORS['FPV_02'], ls='-')}
letters = iter('abcdefghijkl')
for col, sub in enumerate(VARS_SHADE):
    for row, (d, lay) in enumerate(((surf_trio, 'Surface'), (layer_trio, 'Layer mean'))):
        ax = axes[row, col]
        for s in TRIO:
            if sub in d[s].columns:
                ax.plot(d[s].index, d[s][sub], lw=1.2, label=TRIO_LABEL[s], **styles[s])
        ax.grid(True, alpha=0.3); ax.tick_params(labelsize=8)
        add_panel_letter(ax, f'({next(letters)}) {lay}', fontsize=8)
        if row == 0:
            ax.set_title(label_for(sub), fontsize=9)
for ax in axes[-1]:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
h, l = axes[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc='lower center', ncol=3, fontsize=8, frameon=False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
save_fig(fig, '12_shading_only_timeseries')
plt.show()

In [ ]:
# ── Depth profiles: Baseline | Shading only | Δ, and Shading only | FPV_02 | Δ ─────────
shade_profiles = {SHADE: {}}
for sub in ALGAE:
    prof = compute_depth_profile(runs_shade[SHADE], sub, perim_da)
    if prof is not None:
        shade_profiles[SHADE][sub] = prof
chl = None
for sp, ratio in CHLA_C.items():
    if sp in shade_profiles[SHADE]:
        c = shade_profiles[SHADE][sp] * ratio * 1000.0
        chl = c if chl is None else chl.fillna(0) + c.fillna(0)
prof_trio = {'Baseline': {'Chlfa': chlfa_profiles['Baseline']}, SHADE: {'Chlfa': chl},
             'FPV_02': {'Chlfa': chlfa_profiles['FPV_02']}}

# (1) shading effect: Baseline vs Shading only
plot_depth_panel_3col(prof_trio, 'Chlfa', scenario=SHADE, cmap='YlGn',
                      surface_mid_series=surface_mid_depth_series,
                      save_name='12_depth_Chlfa_baseline_vs_shading_only')
# (2) hydrodynamic effect: Shading only vs FPV_02 (Shading only plays the reference role)
prof_hydro = {'Baseline': prof_trio[SHADE], 'FPV_02': prof_trio['FPV_02']}
plot_depth_panel_3col(prof_hydro, 'Chlfa', scenario='FPV_02', cmap='YlGn',
                      surface_mid_series=surface_mid_depth_series,
                      save_name='12_depth_Chlfa_shading_only_vs_fpv02', ref_name='Shading only')

In [ ]:
# ── Surface maps (snapshot dates): three runs on top, three differences below ──
# Each difference panel states the subtraction and what it isolates:
#   (d) Shading only − Baseline : FPV_02 light & wind reduction alone (same hydrodynamics)
#   (e) FPV_02 − Shading only   : FPV_02 hydrodynamics alone (same light & wind)
#   (f) FPV_02 − Baseline       : full FPV_02 effect = (d) + (e)
DIFF_MEANING = {
    (SHADE, 'Baseline'): 'effect of reduced light & wind\n(same hydrodynamics)',
    ('FPV_02', SHADE):   'effect of FPV_02 hydrodynamics\n(same light & wind)',
    ('FPV_02', 'Baseline'): 'full FPV_02 effect = (d) + (e)',
}


def plot_shade_maps(key, date_str, save_name=None):
    f = {s: derived_surface_at_date(rs[s], key, date_str).values
         for rs in (runs, runs_shade) for s in rs if s in TRIO}
    D = [(f[a] - f[b], a, b) for a, b, _ in TRIO_DIFFS]
    vals = np.concatenate([v[np.isfinite(v)] for v in f.values()])
    vmin, vmax = np.nanpercentile(vals, [2, 98])
    dl = np.nanpercentile(np.abs(np.concatenate([d[np.isfinite(d)] for d, _, _ in D])), 98) or 1.0
    cm = plt.get_cmap('YlGn').copy(); cm.set_bad('#dddddd')
    cm_d = plt.get_cmap('RdBu_r').copy(); cm_d.set_bad('#dddddd')
    fig = plt.figure(figsize=(11, 7.6))
    gs = gridspec.GridSpec(2, 4, figure=fig, width_ratios=[1, 1, 1, 0.06], wspace=0.20, hspace=0.30)
    ax0 = None
    for j, s in enumerate(TRIO):
        ax = fig.add_subplot(gs[0, j], sharex=ax0, sharey=ax0); ax0 = ax0 or ax
        im = ax.pcolormesh(np.ma.masked_invalid(f[s].T), cmap=cm, vmin=vmin, vmax=vmax)
        ax.set_aspect('equal'); ax.tick_params(labelsize=7)
        add_panel_letter(ax, f'({"abc"[j]}) {TRIO_LABEL[s]}  {date_str}', fontsize=9)
    for j, (d, a, b) in enumerate(D):
        ax = fig.add_subplot(gs[1, j], sharex=ax0, sharey=ax0)
        imd = ax.pcolormesh(np.ma.masked_invalid(d.T), cmap=cm_d, vmin=-dl, vmax=dl)
        ax.set_aspect('equal'); ax.tick_params(labelsize=7)
        add_panel_letter(ax, f'({"def"[j]}) {TRIO_LABEL[a]} − {TRIO_LABEL[b]}', fontsize=8)
        ax.set_title(DIFF_MEANING[(a, b)], fontsize=8, style='italic')
    fig.colorbar(im, cax=fig.add_subplot(gs[0, -1]), label=label_for(key))
    fig.colorbar(imd, cax=fig.add_subplot(gs[1, -1]), label=f'Δ {label_for(key)}\n(run − reference)')
    fig.text(0.5, 0.005, 'Shading only = run 08: baseline hydrodynamics (FLOW) + FPV_02 light & wind in WAQ. '
             'Red = higher in the first run of the subtraction.', ha='center', fontsize=8)
    if save_name:
        save_fig(fig, save_name)
    plt.show()

for d in SNAPSHOT_DATES:
    plot_shade_maps('Chlfa', d, save_name=f'12_map_Chlfa_shading_only_{d.replace("-", "")}')

In [ ]:
# ── Shading-only effect on its own colour scale: Shading only − Baseline ──────────
# In the maps above all three differences share one colour scale, set by the much larger
# FPV_02 − Shading only difference, so the small shading effect looks white. Here only the
# shading effect is plotted, with its own symmetric scale in discrete steps, plus the same
# difference as % of Baseline. Black outline = FPV_02 footprint (perim_da, 622 columns).
# The scale is shared by all snapshot dates, so the dates can be compared with each other.
from matplotlib.colors import BoundaryNorm

KEY_Z = 'Chlfa'
N_STEPS = 12                                   # number of colour steps (even -> 0 is a boundary)
foot = perim_da.values.astype(float).T         # (N, M) for pcolormesh/contour

fz = {d: {s: derived_surface_at_date(rs[s], KEY_Z, d).values
          for rs in (runs, runs_shade) for s in rs if s in ('Baseline', SHADE)}
      for d in SNAPSHOT_DATES}
dabs = {d: fz[d][SHADE] - fz[d]['Baseline'] for d in SNAPSHOT_DATES}
drel = {d: np.where(fz[d]['Baseline'] > 0, dabs[d] / fz[d]['Baseline'] * 100, np.nan) for d in SNAPSHOT_DATES}


def sym_levels(fields, n=N_STEPS, pct=98):
    v = np.concatenate([np.abs(x[np.isfinite(x)]) for x in fields.values()])
    lim = np.nanpercentile(v, pct) or 1.0
    return np.linspace(-lim, lim, n + 1)


lev_abs, lev_rel = sym_levels(dabs), sym_levels(drel)
cm_z = plt.get_cmap('RdBu_r').copy(); cm_z.set_bad('#dddddd')
norm_abs = BoundaryNorm(lev_abs, cm_z.N, extend='both')
norm_rel = BoundaryNorm(lev_rel, cm_z.N, extend='both')

print(f'Shading only − Baseline, surface {KEY_Z} (FPV_02 footprint vs rest of the lake):')
print(f'  {"Date":<12} {"inside Δ":>10} {"inside %":>9} {"outside Δ":>10} {"outside %":>10}')
for d in SNAPSHOT_DATES:
    inside = perim_da.values.astype(bool)
    wet = np.isfinite(dabs[d])
    ins, out = wet & inside, wet & ~inside
    print(f'  {d:<12} {np.nanmean(dabs[d][ins]):>10.3f} {np.nanmean(drel[d][ins]):>9.2f} '
          f'{np.nanmean(dabs[d][out]):>10.3f} {np.nanmean(drel[d][out]):>10.2f}')

for d in SNAPSHOT_DATES:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.4), sharex=True, sharey=True)
    for ax, field, norm, lab, letter in (
            (axes[0], dabs[d], norm_abs, f'Δ {label_for(KEY_Z)}', '(a)'),
            (axes[1], drel[d], norm_rel, f'Δ {KEY_Z} (% of Baseline)', '(b)')):
        im = ax.pcolormesh(np.ma.masked_invalid(field.T), cmap=cm_z, norm=norm)
        ax.contour(np.arange(foot.shape[1]) + 0.5, np.arange(foot.shape[0]) + 0.5, foot,
                   levels=[0.5], colors='k', linewidths=1.0)
        ax.set_aspect('equal'); ax.tick_params(labelsize=7)
        add_panel_letter(ax, f'{letter} Shading only − Baseline  {d}', fontsize=8)
        fig.colorbar(im, ax=ax, shrink=0.85, label=lab)
    fig.text(0.5, 0.01, 'Blue = lower Chl a in the Shading-only run (FPV_02 light & wind, baseline hydrodynamics). '
             'Black outline = FPV_02 footprint.', ha='center', fontsize=8)
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    save_fig(fig, f'12_map_{KEY_Z}_shading_effect_zoom_{d.replace("-", "")}')
    plt.show()

In [ ]:
# ── Full FPV_02 effect on its own colour scale: FPV_02 − Baseline ──────────────────
# Same layout as the shading-only figure above: (a) difference in µg/L, (b) as % of Baseline,
# own symmetric scale in discrete steps, shared by all snapshot dates.
# Black outline = FPV_02 footprint (perim_da, 622 columns).
# Red = higher Chl a in FPV_02, blue = lower.
from matplotlib.colors import BoundaryNorm

KEY_T = 'Chlfa'
N_STEPS_T = 12                                 # number of colour steps (even -> 0 is a boundary)
foot = perim_da.values.astype(float).T         # (N, M) for pcolormesh/contour

ft = {d: {s: derived_surface_at_date(runs[s], KEY_T, d).values for s in ('Baseline', 'FPV_02')}
      for d in SNAPSHOT_DATES}
tabs = {d: ft[d]['FPV_02'] - ft[d]['Baseline'] for d in SNAPSHOT_DATES}
trel = {d: np.where(ft[d]['Baseline'] > 0, tabs[d] / ft[d]['Baseline'] * 100, np.nan) for d in SNAPSHOT_DATES}


def sym_levels_t(fields, n=N_STEPS_T, pct=98):
    v = np.concatenate([np.abs(x[np.isfinite(x)]) for x in fields.values()])
    lim = np.nanpercentile(v, pct) or 1.0
    return np.linspace(-lim, lim, n + 1)


cm_t = plt.get_cmap('RdBu_r').copy(); cm_t.set_bad('#dddddd')
norm_tabs = BoundaryNorm(sym_levels_t(tabs), cm_t.N, extend='both')
norm_trel = BoundaryNorm(sym_levels_t(trel), cm_t.N, extend='both')

print(f'FPV_02 − Baseline, surface {KEY_T} (FPV_02 footprint vs rest of the lake):')
print(f'  {"Date":<12} {"inside Δ":>10} {"inside %":>9} {"outside Δ":>10} {"outside %":>10}')
inside = perim_da.values.astype(bool)
for d in SNAPSHOT_DATES:
    wet = np.isfinite(tabs[d])
    ins, out = wet & inside, wet & ~inside
    print(f'  {d:<12} {np.nanmean(tabs[d][ins]):>10.3f} {np.nanmean(trel[d][ins]):>9.2f} '
          f'{np.nanmean(tabs[d][out]):>10.3f} {np.nanmean(trel[d][out]):>10.2f}')

for d in SNAPSHOT_DATES:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.4), sharex=True, sharey=True)
    for ax, field, norm, lab, letter in (
            (axes[0], tabs[d], norm_tabs, f'Δ {label_for(KEY_T)}', '(a)'),
            (axes[1], trel[d], norm_trel, f'Δ {KEY_T} (% of Baseline)', '(b)')):
        im = ax.pcolormesh(np.ma.masked_invalid(field.T), cmap=cm_t, norm=norm)
        ax.contour(np.arange(foot.shape[1]) + 0.5, np.arange(foot.shape[0]) + 0.5, foot,
                   levels=[0.5], colors='k', linewidths=1.0)
        ax.set_aspect('equal'); ax.tick_params(labelsize=7)
        add_panel_letter(ax, f'{letter} FPV_02 − Baseline  {d}', fontsize=8)
        fig.colorbar(im, ax=ax, shrink=0.85, label=lab)
    fig.text(0.5, 0.01, 'Red = higher, blue = lower Chl a in FPV_02 (FPV_02 hydrodynamics + light & wind). '
             'Black outline = FPV_02 footprint.', ha='center', fontsize=8)
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    save_fig(fig, f'12_map_{KEY_T}_fpv02_effect_zoom_{d.replace("-", "")}')
    plt.show()

In [ ]:
# ── Why is the shading effect not uniform on 15 June? Check the forcing, then the wind ──
# Part A: is the light and wind reduction really uniform under the array? Maps of the ratio
#         FPV_02 / noFPV of the WAQ radiation and wind forcing (surface segment, K=K_TOP).
# Part B: wind speed and direction from the FLOW meteo input (WAQ only has wind speed).
#         The algae on a given day carry the history of the water that moved under the array
#         over the previous days, so the wind is shown for a window of days before the map date.
FACTOR_DATE = '2023-06-15'
WIN_DAYS    = 7                     # days before FACTOR_DATE (inclusive) for the window means
WIND_BIN = {                        # WAQ wind forcing (speed only) — check the names/folder
    'noFPV': 'waq-wind-noFPV_remapped_upper.bin',
    '02':    'waq-wind-02towaq_fpv41_nogaps_remapped_upper.bin',
}
# FLOW uniform-wind file (.wnd: time [min after FLOW reference date], speed [m/s], direction [deg]).
# Set to None to skip Part B.
FLOW_WND_FILE = None                # e.g. os.path.join(BIN_DIR, '<flow folder>', 'torun.wnd')
FLOW_REFDATE  = None                # FLOW reference date (Itdate in the .mdf), e.g. '2022-09-09'

t_day0 = pd.Timestamp(FACTOR_DATE)
t_win0 = t_day0 - pd.Timedelta(days=WIN_DAYS - 1)
t_end  = t_day0 + pd.Timedelta(days=1)


def seg_memmap(path, nseg=N_SEG):
    rec = np.dtype([('t', '<i4'), ('v', '<f4', (nseg,))])
    return np.memmap(path, dtype=rec, mode='r', shape=(os.path.getsize(path) // rec.itemsize,))


def window_mean(path, t0, t1):
    """Mean per segment of all records with t0 <= time < t1 (record header = seconds after T0)."""
    mm = seg_memmap(path)
    t = T0 + pd.to_timedelta(np.asarray(mm['t']).astype('int64'), unit='s')
    sel = np.where((t >= t0) & (t < t1))[0]
    if sel.size == 0:
        raise ValueError(f'{os.path.basename(path)}: no records between {t0} and {t1}')
    return np.asarray(mm['v'][sel]).astype(float).mean(axis=0), sel.size


# compact segment -> (M, N) of the surface layer, as used for perim_da
old_to_new = {int(o): i for i, o in enumerate(new_to_old)}
surf_idx = np.full(segnum.shape[:2], -1)
for m_ in range(segnum.shape[0]):
    for n_ in range(segnum.shape[1]):
        v = segnum[m_, n_, K_TOP]
        if np.isfinite(v):
            surf_idx[m_, n_] = old_to_new.get(int(v) - 1, -1)


def to_mn(vec):
    out = np.full(surf_idx.shape, np.nan)
    ok = surf_idx >= 0
    out[ok] = vec[surf_idx[ok]]
    return out


# ── Part A: forcing ratios ────────────────────────────────────────────────
ratios = {}
for var, files in (('Radiation', RAD_BIN), ('Wind speed', WIND_BIN)):
    for lab, (t0, t1) in ((f'{FACTOR_DATE}', (t_day0, t_end)),
                          (f'{t_win0.date()} – {FACTOR_DATE}', (t_win0, t_end))):
        f02, n02 = window_mean(os.path.join(BIN_DIR, files['02']), t0, t1)
        f00, n00 = window_mean(os.path.join(BIN_DIR, files['noFPV']), t0, t1)
        ratios[(var, lab)] = to_mn(np.where(f00 > 1e-6, f02 / f00, np.nan))
        print(f'{var:<11} {lab:<25} records used: FPV_02 {n02}, noFPV {n00}')

inside = perim_da.values.astype(bool)
print('\nRatio FPV_02 / noFPV inside the FPV_02 footprint (values rounded to 0.01, cell counts):')
for (var, lab), r in ratios.items():
    vals, counts = np.unique(np.round(r[inside & np.isfinite(r)], 2), return_counts=True)
    print(f'  {var:<11} {lab:<25} ' + ', '.join(f'{v:.2f}×{c}' for v, c in zip(vals, counts)))
    out = r[~inside & np.isfinite(r)]
    print(f'  {"":<11} {"outside footprint":<25} min {out.min():.3f}, max {out.max():.3f}')

foot = perim_da.values.astype(float).T
cm_r = plt.get_cmap('viridis', 8).copy(); cm_r.set_bad('#dddddd')
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True, sharey=True)
for ax, ((var, lab), r), letter in zip(axes.flatten(), ratios.items(), 'abcd'):
    im = ax.pcolormesh(np.ma.masked_invalid(r.T), cmap=cm_r, vmin=0.6, vmax=1.0)
    ax.contour(np.arange(foot.shape[1]) + 0.5, np.arange(foot.shape[0]) + 0.5, foot,
               levels=[0.5], colors='r', linewidths=1.0)
    ax.set_aspect('equal'); ax.tick_params(labelsize=7)
    add_panel_letter(ax, f'({letter}) {var}: FPV_02 / noFPV\n{lab}', fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.8, label='Ratio FPV_02 / noFPV')
fig.text(0.5, 0.01, 'WAQ forcing at the surface segment (K=K_TOP). 1 = no reduction. '
         'Red outline = FPV_02 footprint.', ha='center', fontsize=8)
plt.tight_layout(rect=(0, 0.03, 1, 1))
save_fig(fig, f'12_forcing_ratio_maps_{FACTOR_DATE.replace("-", "")}')
plt.show()

# ── Part B: wind direction from the FLOW meteo input ──────────────────────
if FLOW_WND_FILE is None:
    print('Part B skipped: set FLOW_WND_FILE and FLOW_REFDATE (WAQ forcing has wind speed only).')
else:
    w = pd.read_csv(FLOW_WND_FILE, sep=r'\s+', header=None, comment='*',
                    names=['tmin', 'speed', 'dir'], usecols=[0, 1, 2])
    w.index = pd.Timestamp(FLOW_REFDATE) + pd.to_timedelta(w['tmin'], unit='min')
    ww = w.loc[t_win0:t_end - pd.Timedelta(seconds=1)]
    # Nautical convention assumed: direction the wind comes FROM, clockwise from north.
    u = -ww['speed'] * np.sin(np.deg2rad(ww['dir']))   # eastward component
    v = -ww['speed'] * np.cos(np.deg2rad(ww['dir']))   # northward component
    vec_from = (np.rad2deg(np.arctan2(-u.mean(), -v.mean())) + 360) % 360
    day = ww.loc[FACTOR_DATE]
    ud = -day['speed'] * np.sin(np.deg2rad(day['dir'])); vd = -day['speed'] * np.cos(np.deg2rad(day['dir']))
    day_from = (np.rad2deg(np.arctan2(-ud.mean(), -vd.mean())) + 360) % 360
    print(f'Window {t_win0.date()} – {FACTOR_DATE}: mean speed {ww["speed"].mean():.2f} m/s, '
          f'vector-mean direction FROM {vec_from:.0f}° (steadiness {np.hypot(u.mean(), v.mean()) / ww["speed"].mean():.2f})')
    print(f'{FACTOR_DATE}: mean speed {day["speed"].mean():.2f} m/s, vector-mean direction FROM {day_from:.0f}°')

    fig = plt.figure(figsize=(12, 4.2))
    gs = gridspec.GridSpec(2, 2, figure=fig, width_ratios=[2.2, 1], hspace=0.15)
    ax1 = fig.add_subplot(gs[0, 0]); ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
    ax1.plot(ww.index, ww['speed'], color='k', lw=1)
    ax1.set_ylabel('Speed (m/s)', fontsize=9); ax1.grid(True, alpha=0.3)
    ax2.scatter(ww.index, ww['dir'], s=6, c=ww['speed'], cmap='viridis')
    ax2.set_ylim(0, 360); ax2.set_yticks([0, 90, 180, 270, 360], ['N', 'E', 'S', 'W', 'N'])
    ax2.set_ylabel('From', fontsize=9); ax2.grid(True, alpha=0.3)
    for ax in (ax1, ax2):
        ax.axvspan(t_day0, t_end, color='orange', alpha=0.2)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    add_panel_letter(ax1, '(a) FLOW wind speed', fontsize=8)
    add_panel_letter(ax2, '(b) FLOW wind direction (from)', fontsize=8)
    axr = fig.add_subplot(gs[:, 1], projection='polar')
    axr.set_theta_zero_location('N'); axr.set_theta_direction(-1)
    bins = np.deg2rad(np.arange(0, 361, 22.5))
    h, _ = np.histogram(np.deg2rad(ww['dir'] % 360), bins=bins, weights=ww['speed'])
    axr.bar(bins[:-1] + np.deg2rad(11.25), h / h.sum() * 100, width=np.deg2rad(22.5),
            color='#1f77b4', edgecolor='k', lw=0.3)
    axr.set_title(f'(c) Wind rose, speed-weighted (% , from)\n{t_win0.date()} – {FACTOR_DATE}', fontsize=8)
    axr.tick_params(labelsize=7)
    save_fig(fig, f'12_flow_wind_{FACTOR_DATE.replace("-", "")}')
    plt.show()

In [ ]:
# ── Currents under and around the array: velocity vectors from the WAQ hydrodynamics ──
# Run 08 and the Baseline share the same (baseline) FLOW hydrodynamics, so these currents are the ones
# that carried the algae in both runs. Velocity per exchange = flow (.flo, m³/s) / area (.are, m²);
# the .poi gives the two segments of each exchange. Arrows are drawn in model-grid space (M to the
# right, N up), like all maps in this notebook — directions are relative to the grid, not to north.
# Background: Shading only − Baseline surface Chl a on VEL_DATE.
from matplotlib.colors import BoundaryNorm
HYD_DIR   = os.path.join(BIN_DIR, '08_baselinewithSRWSfrom02')   # folder with the run's com-*.poi/.are/.flo
HYD_BASE  = 'com-WQexport_clean'
VEL_DATE  = '2023-06-15'
VEL_DAYS  = 7                  # window for the residual (time-mean) current, days up to VEL_DATE
VEL_LAYER = 'surface'          # 'surface' (K=K_TOP) or 'column' (flow-weighted mean over K_TOP..K_BOT)
Q_STEP    = 2                  # draw every Q_STEP-th arrow in M and N
Q_REF     = None               # reference arrow (m/s); None = automatic (95th-percentile speed, rounded)

t_v1 = pd.Timestamp(VEL_DATE) + pd.Timedelta(days=1)
t_v0 = t_v1 - pd.Timedelta(days=VEL_DAYS)

# ── exchanges -> grid positions ───────────────────────────────────────────
poi = np.fromfile(os.path.join(HYD_DIR, HYD_BASE + '.poi'), dtype='<i4').reshape(-1, 4)[:, :2]
NOQ = len(poi)
compact = poi.max() <= N_SEG
print(f'{NOQ} exchanges; pointer numbering: {"trimmed (compact)" if compact else "full grid"}')

old_pos = {}
for m_, n_, k_ in np.argwhere(np.isfinite(segnum)):
    old_pos[int(segnum[m_, n_, k_])] = (m_, n_, k_)


def seg_pos(s):
    if s <= 0:
        return None
    return old_pos.get(int(new_to_old[s - 1]) + 1 if compact else int(s))


P = np.array([(seg_pos(a) or (-1, -1, -1)) + (seg_pos(b) or (-1, -1, -1)) for a, b in poi])
ok = (P[:, 0] >= 0) & (P[:, 3] >= 0) & (P[:, 2] == P[:, 5])          # horizontal, both segments known
dm, dn = P[:, 3] - P[:, 0], P[:, 4] - P[:, 1]
in_m = ok & (np.abs(dm) == 1) & (dn == 0)
in_n = ok & (np.abs(dn) == 1) & (dm == 0)
k_ok = (P[:, 2] == K_TOP) if VEL_LAYER == 'surface' else ((P[:, 2] >= K_TOP) & (P[:, 2] <= K_BOT))
print(f'  horizontal exchanges used: {int((in_m & k_ok).sum())} along M, {int((in_n & k_ok).sum())} along N '
      f'({VEL_LAYER}); skipped {int((~ok).sum())} vertical/boundary/unmapped')


def exch_memmap(path):
    rec = np.dtype([('t', '<i4'), ('v', '<f4', (NOQ,))])
    return np.memmap(path, dtype=rec, mode='r', shape=(os.path.getsize(path) // rec.itemsize,))


def window_sum(path, t0, t1):
    mm = exch_memmap(path)
    t = T0 + pd.to_timedelta(np.asarray(mm['t']).astype('int64'), unit='s')
    sel = np.where((t >= t0) & (t < t1))[0]
    if sel.size == 0:
        raise ValueError(f'{os.path.basename(path)}: no records between {t0} and {t1}')
    return np.asarray(mm['v'][sel]).astype(float).mean(axis=0), sel.size


def cell_velocity(t0, t1):
    """Time-mean velocity per (M, N) cell: Σflow / Σarea on each face (over the layers used),
    signed in the +M / +N direction, then averaged over the two faces of the cell."""
    q, nq = window_sum(os.path.join(HYD_DIR, HYD_BASE + '.flo'), t0, t1)
    a, na = window_sum(os.path.join(HYD_DIR, HYD_BASE + '.are'), t0, t1)
    shape = segnum.shape[:2]
    out = []
    for sel, d in ((in_m & k_ok, dm), (in_n & k_ok, dn)):
        face = pd.DataFrame({'m1': P[sel, 0], 'n1': P[sel, 1], 'm2': P[sel, 3], 'n2': P[sel, 4],
                             'q': q[sel] * d[sel], 'a': a[sel]})
        face = face.groupby(['m1', 'n1', 'm2', 'n2'], as_index=False)[['q', 'a']].sum()
        face = face[face['a'] > 0]
        vel = (face['q'] / face['a']).values
        s, c = np.zeros(shape), np.zeros(shape)
        for col_m, col_n in (('m1', 'n1'), ('m2', 'n2')):
            np.add.at(s, (face[col_m].values, face[col_n].values), vel)
            np.add.at(c, (face[col_m].values, face[col_n].values), 1)
        out.append(np.where(c > 0, s / np.maximum(c, 1), np.nan))
    print(f'  {t0.date()} – {(t1 - pd.Timedelta(seconds=1)).date()}: {nq} flow and {na} area records')
    return out[0], out[1]                      # u along +M, v along +N (m/s)


vel = {f'{VEL_DATE}': cell_velocity(pd.Timestamp(VEL_DATE), t_v1),
       f'{t_v0.date()} – {VEL_DATE} (mean)': cell_velocity(t_v0, t_v1)}

# ── background: shading effect on VEL_DATE (same scale as the shading-effect maps if they ran) ──
bg_f = {s: derived_surface_at_date(rs[s], 'Chlfa', VEL_DATE).values
        for rs in (runs, runs_shade) for s in rs if s in ('Baseline', SHADE)}
bg = bg_f[SHADE] - bg_f['Baseline']
try:
    bg_norm = norm_abs                         # from the shading-effect cell, if it ran
except NameError:
    lim = np.nanpercentile(np.abs(bg[np.isfinite(bg)]), 98) or 1.0
    bg_norm = BoundaryNorm(np.linspace(-lim, lim, 13), 256, extend='both')

foot = perim_da.values.astype(float).T
cm_v = plt.get_cmap('RdBu_r').copy(); cm_v.set_bad('#dddddd')
sp_all = np.concatenate([np.hypot(u, v)[np.isfinite(np.hypot(u, v))] for u, v in vel.values()])
sp95 = np.nanpercentile(sp_all, 95)
if Q_REF is None:
    Q_REF = float(f'{sp95:.1g}')
q_scale = sp95 / (0.9 * Q_STEP)                 # 95th-percentile speed -> 0.9 × arrow spacing (grid cells)
M_, N_ = np.meshgrid(np.arange(segnum.shape[0]) + 0.5, np.arange(segnum.shape[1]) + 0.5, indexing='ij')
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharex=True, sharey=True)
for ax, (lab, (u, v)), letter in zip(axes, vel.items(), 'ab'):
    im = ax.pcolormesh(np.ma.masked_invalid(bg.T), cmap=cm_v, norm=bg_norm)
    ax.contour(np.arange(foot.shape[1]) + 0.5, np.arange(foot.shape[0]) + 0.5, foot,
               levels=[0.5], colors='k', linewidths=1.0)
    s_ = (slice(None, None, Q_STEP), slice(None, None, Q_STEP))
    Q = ax.quiver(M_[s_], N_[s_], u[s_], v[s_], angles='xy', scale_units='xy',
                  scale=q_scale, width=0.003, color='k')
    ax.quiverkey(Q, 0.80, 1.03, Q_REF, f'{Q_REF} m/s', labelpos='E', fontproperties={'size': 8})
    ax.set_aspect('equal'); ax.tick_params(labelsize=7)
    add_panel_letter(ax, f'({letter}) {VEL_LAYER} current, {lab}', fontsize=8)
    sp = np.hypot(u, v)
    print(f'  ({letter}) {lab}: speed median {np.nanmedian(sp):.4f}, 95th pct {np.nanpercentile(sp, 95):.4f} m/s')
fig.colorbar(im, ax=axes, shrink=0.85, label=f'Δ Chl a, Shading only − Baseline (µg/L), {VEL_DATE}')
fig.text(0.45, 0.01, 'Arrows: Baseline (= run 08) hydrodynamics, flow/area from the WAQ .flo/.are, in grid space '
         '(M right, N up). Black outline = FPV_02 footprint.', ha='center', fontsize=8)
save_fig(fig, f'12_currents_{VEL_LAYER}_{VEL_DATE.replace("-", "")}')
plt.show()

In [ ]:
# ── Check the current calculation before interpreting it ─────────────────────────
# Uses poi, P, in_m, in_n, ok, NOQ, HYD_DIR, HYD_BASE, t_v0, t_v1 from the current-vector cell.
NOQ3 = 16046                    # 'exchanges in direction 3' in the .inp
NOQ2 = 16840                    # 'exchanges in direction 2' in the .inp
NOQ1 = NOQ - NOQ2 - NOQ3
print(f'NOQ = {NOQ}  ->  direction 1: {NOQ1}, direction 2: {NOQ2}, direction 3: {NOQ3}')

# 1) Does the grid-based classification agree with DELWAQ's own direction blocks?
blocks = {'dir 1': slice(0, NOQ1), 'dir 2': slice(NOQ1, NOQ1 + NOQ2), 'dir 3': slice(NOQ1 + NOQ2, NOQ)}
vert = ok.copy(); vert[:] = False
vert[(P[:, 0] >= 0) & (P[:, 3] >= 0) & (P[:, 0] == P[:, 3]) & (P[:, 1] == P[:, 4])] = True
print('\nClassification of each DELWAQ direction block (counts):')
print(f'  {"block":<6} {"along M":>8} {"along N":>8} {"vertical":>9} {"other":>7}')
for name, sl in blocks.items():
    m_, n_, v_ = in_m[sl].sum(), in_n[sl].sum(), vert[sl].sum()
    print(f'  {name:<6} {m_:>8} {n_:>8} {v_:>9} {sl.stop - sl.start - m_ - n_ - v_:>7}')

# 2) Flows and areas per direction at the surface layer and below, over the window
def window_stats(path, t0, t1):
    mm = exch_memmap(path)
    t = T0 + pd.to_timedelta(np.asarray(mm['t']).astype('int64'), unit='s')
    sel = np.where((t >= t0) & (t < t1))[0]
    X = np.asarray(mm['v'][sel]).astype(float)
    return X, t[sel]

Qw, tq = window_stats(os.path.join(HYD_DIR, HYD_BASE + '.flo'), t_v0, t_v1)
Aw, _  = window_stats(os.path.join(HYD_DIR, HYD_BASE + '.are'), t_v0, t_v1)
print(f'\nRecords in window: {len(tq)}, first {tq[0]}, last {tq[-1]}, step {tq[1] - tq[0] if len(tq) > 1 else "n/a"}')
print('\n|velocity| = |flow|/area per exchange, over all records in the window (m/s):')
print(f'  {"set":<22} {"n exch":>7} {"area>0 %":>9} {"median":>9} {"95th":>9} {"time-mean u":>12}')
for kname, kk in (('K_TOP', K_TOP), ('K_TOP+1', K_TOP + 1), ('K_TOP+3', K_TOP + 3)):
    for dname, sel in (('along M', in_m), ('along N', in_n)):
        s = sel & (P[:, 2] == kk)
        if not s.any():
            continue
        q, a = Qw[:, s], Aw[:, s]
        with np.errstate(invalid='ignore', divide='ignore'):
            v = np.where(a > 0, q / a, np.nan)
        d = (P[s, 3] - P[s, 0]) if dname == 'along M' else (P[s, 4] - P[s, 1])
        print(f'  {kname + " " + dname:<22} {s.sum():>7} {100 * (a > 0).mean():>9.1f} '
              f'{np.nanmedian(np.abs(v)):>9.5f} {np.nanpercentile(np.abs(v), 95):>9.5f} '
              f'{np.nanmean(v * d):>12.5f}')

In [ ]:
# ── Why does the NW interior respond most? Depth vs distance from unshaded water ─────────
# Inside the footprint the forcing is uniform (0.65), so differences in Δ% come from the water itself.
# Two candidate controls, per footprint cell:
#   - column depth: number of wet layers (K_TOP..K_BOT) × nominal layer thickness (top layer nominal, approx.)
#   - distance (grid cells) to the nearest wet cell outside the footprint (dilution by unshaded water)
from scipy import ndimage

DD_DATE = '2023-06-15'

# Δ% Shading only − Baseline, surface Chl a
f_b = derived_surface_at_date(runs['Baseline'], 'Chlfa', DD_DATE).values
f_s = derived_surface_at_date(runs_shade[SHADE], 'Chlfa', DD_DATE).values
d_pct = np.where(f_b > 0, (f_s - f_b) / f_b * 100, np.nan)

# column depth from the wet layers on that day
wet3 = runs['Baseline']['OXY'].sel(K=slice(K_TOP, K_BOT)).sel(time=pd.Timestamp(DD_DATE), method='nearest').notnull()
thk = xr.DataArray([THICKNESS_M[int(k)] for k in wet3.K.values], coords={'K': wet3.K}, dims='K')
depth = (wet3 * thk).sum('K').where(wet3.any('K')).transpose('M', 'N').values

# distance to the nearest wet cell outside the footprint (in grid cells)
foot_b = perim_da.values.astype(bool)
wet2 = np.isfinite(d_pct)
dist = ndimage.distance_transform_edt(~(wet2 & ~foot_b))      # 0 on unshaded wet cells
dist = np.where(foot_b & wet2, dist, np.nan)

sel = foot_b & wet2 & np.isfinite(depth)
x_dep, x_dis, y = depth[sel], dist[sel], d_pct[sel]
r_dep = np.corrcoef(x_dep, y)[0, 1]
r_dis = np.corrcoef(x_dis, y)[0, 1]
print(f'{DD_DATE}, {sel.sum()} footprint cells: Δ% range {y.min():.2f} to {y.max():.2f}')
print(f'  Pearson r (Δ% vs column depth):            {r_dep:+.2f}')
print(f'  Pearson r (Δ% vs distance to unshaded water): {r_dis:+.2f}')
print(f'  depth range {x_dep.min():.2f}–{x_dep.max():.2f} m, distance range {x_dis.min():.1f}–{x_dis.max():.1f} cells')
print('  (negative r = larger depth/distance goes with a stronger reduction)')

foot = perim_da.values.astype(float).T
fig = plt.figure(figsize=(13, 7.5))
gs = gridspec.GridSpec(2, 3, figure=fig, height_ratios=[1, 0.8], hspace=0.30, wspace=0.55)
maps = ((d_pct, f'Δ Chl a (%), Shading only − Baseline', 'RdBu_r', None),
        (depth, 'Column depth (m)', 'viridis_r', None),
        (np.where(wet2, dist, np.nan), 'Distance to unshaded water (cells)', 'magma_r', None))
ax0 = None
for j, (fld, lab, cmap, _) in enumerate(maps):
    ax = fig.add_subplot(gs[0, j], sharex=ax0, sharey=ax0); ax0 = ax0 or ax
    cm = plt.get_cmap(cmap).copy(); cm.set_bad('#dddddd')
    if j == 0:
        lim = np.nanpercentile(np.abs(fld), 98)
        im = ax.pcolormesh(np.ma.masked_invalid(fld.T), cmap=cm, vmin=-lim, vmax=lim)
    else:
        im = ax.pcolormesh(np.ma.masked_invalid(fld.T), cmap=cm)
    ax.contour(np.arange(foot.shape[1]) + 0.5, np.arange(foot.shape[0]) + 0.5, foot,
               levels=[0.5], colors='k', linewidths=1.0)
    ax.set_aspect('equal'); ax.tick_params(labelsize=7)
    add_panel_letter(ax, f'({"abc"[j]}) {lab}', fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.8)
for j, (x, lab, c, clab, r) in enumerate(((x_dep, 'Column depth (m)', x_dis, 'distance (cells)', r_dep),
                                          (x_dis, 'Distance to unshaded water (cells)', x_dep, 'depth (m)', r_dis))):
    ax = fig.add_subplot(gs[1, j])
    sc = ax.scatter(x, y, c=c, s=8, cmap='viridis')
    ax.set_xlabel(lab, fontsize=9); ax.set_ylabel('Δ Chl a (%)', fontsize=9)
    ax.grid(True, alpha=0.3); ax.tick_params(labelsize=8)
    add_panel_letter(ax, f'({"de"[j]}) footprint cells, r = {r:+.2f}', fontsize=8)
    fig.colorbar(sc, ax=ax, label=f'colour: {clab}')
fig.text(0.5, 0.01, f'{DD_DATE}, surface (K=K_TOP). Black outline = FPV_02 footprint. '
         'Depth = wet layers × nominal thickness (approximate for the top layer).', ha='center', fontsize=8)
save_fig(fig, f'12_shading_effect_vs_depth_distance_{DD_DATE.replace("-", "")}')
plt.show()

In [ ]:
# ── Shading effect by layer: surface and sub-surface layers on one date ─────────────────
# Rows = layers, columns = (a) Baseline Chl a, (b) Δ Shading only − Baseline (µg/L), (c) Δ (%).
# Continuous colour scales covering all values (no percentile clipping), shared by all rows,
# so the layers can be compared directly. Black outline = FPV_02 footprint.
LAY_DATE = '2023-06-15'
LAYERS   = [K_TOP, K_TOP + 1, K_TOP + 2]        # K indices (K_TOP = surface layer)


def chla_layer_at_date(ds, k, target_date):
    """Daily-mean Chl a (µg/L) at layer k on target_date = Σ biomass × CHLACALG × 1000."""
    t0 = pd.Timestamp(target_date)
    out = None
    for s in ALGAE:
        if s not in ds.data_vars:
            continue
        da = ds[s].sel(K=k).sel(time=slice(t0, t0 + pd.Timedelta(hours=23, minutes=59))).mean('time')
        out = da * CHLA_C[s] * 1000.0 if out is None else out + da * CHLA_C[s] * 1000.0
    return out.transpose('M', 'N').values


lay = {}
for k in LAYERS:
    b = chla_layer_at_date(runs['Baseline'], k, LAY_DATE)
    s = chla_layer_at_date(runs_shade[SHADE], k, LAY_DATE)
    lay[k] = (b, s - b, np.where(b > 0, (s - b) / b * 100, np.nan))

inside = perim_da.values.astype(bool)
print(f'Shading only − Baseline, Chl a on {LAY_DATE}, by layer (footprint vs rest of the lake):')
print(f'  {"Layer":<10} {"Base in":>8} {"Base out":>9} {"Δ in":>8} {"Δ% in":>8} {"Δ out":>8} {"Δ% out":>8} {"wet cells":>10}')
for k, (b, d, p) in lay.items():
    w = np.isfinite(p)
    i_, o_ = w & inside, w & ~inside
    print(f'  K={k:<8} {np.nanmean(b[i_]):>8.2f} {np.nanmean(b[o_]):>9.2f} {np.nanmean(d[i_]):>8.3f} '
          f'{np.nanmean(p[i_]):>8.2f} {np.nanmean(d[o_]):>8.3f} {np.nanmean(p[o_]):>8.2f} {w.sum():>10}')


def full_range(arrs, symmetric=False):
    v = np.concatenate([a[np.isfinite(a)] for a in arrs])
    if symmetric:
        lim = np.abs(v).max() or 1.0
        return -lim, lim
    return v.min(), v.max()


rng_b = full_range([x[0] for x in lay.values()])
rng_d = full_range([x[1] for x in lay.values()], symmetric=True)
rng_p = full_range([x[2] for x in lay.values()], symmetric=True)
foot = perim_da.values.astype(float).T
cm_b = plt.get_cmap('YlGn').copy(); cm_b.set_bad('#dddddd')
cm_d = plt.get_cmap('RdBu_r').copy(); cm_d.set_bad('#dddddd')

fig, axes = plt.subplots(len(LAYERS), 3, figsize=(12, 3.6 * len(LAYERS)), sharex=True, sharey=True,
                         squeeze=False)
for r, k in enumerate(LAYERS):
    for c, (fld, cm, (lo, hi), lab) in enumerate((
            (lay[k][0], cm_b, rng_b, 'Baseline Chl a (µg/L)'),
            (lay[k][1], cm_d, rng_d, 'Δ Chl a (µg/L)'),
            (lay[k][2], cm_d, rng_p, 'Δ Chl a (% of Baseline)'))):
        ax = axes[r, c]
        im = ax.pcolormesh(np.ma.masked_invalid(fld.T), cmap=cm, vmin=lo, vmax=hi)
        ax.contour(np.arange(foot.shape[1]) + 0.5, np.arange(foot.shape[0]) + 0.5, foot,
                   levels=[0.5], colors='k', linewidths=1.0)
        ax.set_aspect('equal'); ax.tick_params(labelsize=7)
        name = 'surface' if k == K_TOP else f'{k - K_TOP} below surface'
        add_panel_letter(ax, f'({"abcdefghijkl"[3 * r + c]}) K={k} ({name})', fontsize=8)
        fig.colorbar(im, ax=ax, shrink=0.85, label=lab)
fig.suptitle(f'Shading only − Baseline, {LAY_DATE}', fontsize=10)
fig.text(0.5, 0.005, 'Δ = Shading only (FPV_02 light & wind, baseline hydrodynamics) − Baseline. '
         'Colour ranges cover all values and are shared by all layers.', ha='center', fontsize=8)
plt.tight_layout(rect=(0, 0.02, 1, 0.98))
save_fig(fig, f'12_shading_effect_by_layer_{LAY_DATE.replace("-", "")}')
plt.show()